# MineToday V38.1 — Hybrid34-Locked Curriculum XLM-R Semantic Membership & Pairwise Judge

**Goal:** exceed the public Hybrid34 anchor (`0.66006`) with a new semantic information source while preserving every public-proven ranking decision unless a curriculum-trained XLM-R judge earns a calibrated residual override.

V38.1 is deliberately **not** “V38 with more epochs / lower thresholds / XLM-R Large.” The V38 run showed that a pretrained XLM-R Base thrown directly at rare Hybrid34 residual errors learned only a weak signal and produced zero calibration-approved actions. V38.1 changes the learning problem instead:

1. **Stage A — broad MineToday semantic relevance adaptation.** XLM-R Base learns user↔module relevance from all 17 organizer modules for each fit-side user using binary relevance + raw relevance-score regression.
2. **Stage B — hard pairwise curriculum.** The Stage-A encoder initializes a directional A-vs-B semantic judge trained on informative non-tie pairs with exact pair reversal. Only the upper six XLM-R transformer layers are adapted here.
3. **Stage C — frozen semantic residual calibration.** The semantic encoders are frozen. Only small grouped cross-fitted Logistic/Ridge calibrators learn whether semantic disagreement is sufficient to override Hybrid34, using exact signed ΔNDCG as the utility target.

### Frozen competition decisions
- exact deterministic Hybrid34 (`0.66006`) is the immutable Stage-0 anchor;
- S3 is immutable, S1/S2 are excluded, only S0 may be touched;
- rank `1↔2` is hard-locked;
- `ORDER_SAFE` considers only `2↔3`, `3↔4`, `4↔5`;
- `MEMBERSHIP_ATTACK` specializes on `5↔6`, where Top-5 membership leverage is much larger;
- `ATTACK` is the precommitted union of calibrated order + membership actions, with at most one action per user;
- score multisets are preserved exactly by adjacent score-level swaps;
- no XE/Yeti/Qwen/LLM embedding branch, no TAPT/MLM, no static cosine embeddings, no LightGBM meta-router;
- two independent XLM-R Base semantic seeds provide mean + disagreement features;
- fresh AA/AB/AC cohorts are disjoint and used only for accept/reject after all policy rules are frozen;
- sealed template-disjoint D + chronology are rejection-only;
- if any gate fails, `submission_v38_1_primary.csv` is exactly the Hybrid34 anchor and must not consume a Kaggle slot.


In [1]:
# Runtime + preregistered V38.1 configuration
import os, sys, re, json, ast, html, hashlib, random, warnings, time, subprocess, gc, math, copy
from pathlib import Path
from collections import Counter, defaultdict

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ["PYTHONHASHSEED"] = "42"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

V381_SKIP_INSTALL = bool(int(os.getenv("V381_SKIP_INSTALL", "0")))
try:
    import lightgbm as lgb
except Exception:
    if V381_SKIP_INSTALL: raise
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm==4.6.0"])
    import lightgbm as lgb

try:
    import torch, transformers
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
except Exception:
    if V381_SKIP_INSTALL: raise
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.46,<6", "sentencepiece", "safetensors"])
    import torch, transformers
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.sparse import hstack
from scipy.optimize import minimize_scalar
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, StratifiedShuffleSplit, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, mean_absolute_error

warnings.filterwarnings("ignore")

SEED = 42
V381_SMOKE = bool(int(os.getenv("V381_SMOKE", "0")))
V381_STRICT_GEOMETRY = bool(int(os.getenv("V381_STRICT_GEOMETRY", "1")))
GEOMETRY_ATOL = 2.5e-4

# Compatibility aliases used by frozen V25/V30/V31 helpers inherited from V38.
V38_SMOKE = V381_SMOKE
V38_STRICT_GEOMETRY = V381_STRICT_GEOMETRY
V37_SMOKE = V381_SMOKE
V361_SMOKE = V381_SMOKE
V36_SMOKE = V381_SMOKE
V31_SMOKE = V381_SMOKE
V30_SMOKE = V381_SMOKE
V29_SMOKE = V381_SMOKE
V27_SMOKE = V381_SMOKE
V37_STRICT_GEOMETRY = V381_STRICT_GEOMETRY
V361_STRICT_GEOMETRY = V381_STRICT_GEOMETRY
V36_STRICT_GEOMETRY = V381_STRICT_GEOMETRY
V30_STRICT_GEOMETRY = V381_STRICT_GEOMETRY
V29_STRICT_GEOMETRY = V381_STRICT_GEOMETRY
V27_STRICT_GEOMETRY = V381_STRICT_GEOMETRY

N_FOLDS = 2 if V381_SMOKE else 5
ANCHOR_INNER_FOLDS = 2 if V381_SMOKE else 4
INTENT_INNER_FOLDS = 2 if V381_SMOKE else 4
RIDGE_ALPHA = 3.0
REFERENCE_W = 0.45
PUBLIC_CAL045_LB = 0.65639
PUBLIC_CONSENSUS3_LB = 0.65856
PUBLIC_V25_LB = 0.65919
PUBLIC_V30_LB = 0.65963
PUBLIC_V31_LB = 0.65949
PUBLIC_HYBRID34_LB = 0.66006
PUBLIC_LEADER_LB = 0.66142
PUBLIC_LEADER_GAP = PUBLIC_LEADER_LB - PUBLIC_HYBRID34_LB

LGB_ESTIMATORS = 12 if V381_SMOKE else 1200
LGB_EARLY_STOP = 3 if V381_SMOKE else 80
RANKER_ESTIMATORS = 16 if V381_SMOKE else 360
RANKER_LEARNING_RATE = 0.04 if V381_SMOKE else 0.025
ALPHA = 0.16
INTENT_PARENT_STRENGTH = 40.0
INTENT_CHILD_STRENGTH = 30.0
BOOTSTRAPS = 20 if V381_SMOKE else 1200

# Curriculum XLM-R Base. Stage A full adaptation; Stage B upper-six-layer specialization.
XLMR_MODEL_ID = "FacebookAI/xlm-roberta-base"
XLMR_MAX_LENGTH = 96 if V381_SMOKE else 256
STAGEA_EPOCHS = 1
STAGEA_BATCH = 8 if V381_SMOKE else 16
STAGEA_GRAD_ACCUM = 1 if V381_SMOKE else 2
STAGEA_ENCODER_LR = 2.0e-5 if V381_SMOKE else 1.5e-5
STAGEA_HEAD_LR = 1.0e-4 if V381_SMOKE else 8.0e-5
STAGEA_SCORE_WEIGHT = 0.70
STAGEB_EPOCHS = 1
STAGEB_BATCH = 8 if V381_SMOKE else 16
STAGEB_GRAD_ACCUM = 1 if V381_SMOKE else 2
STAGEB_TOP_LAYERS = 6
STAGEB_ENCODER_LR = 1.0e-5 if V381_SMOKE else 8.0e-6
STAGEB_HEAD_LR = 8.0e-5 if V381_SMOKE else 5.0e-5
XLMR_WEIGHT_DECAY = 0.01
XLMR_WARMUP = 0.08
XLMR_GRAD_CLIP = 1.0
SEMANTIC_SEEDS = (271,) if V381_SMOKE else (271,829)
CAL_FRACTION = 0.20
INNER_HYBRID_FOLDS = 2 if V381_SMOKE else 4
ELIGIBLE_POSITIONS = (2,3,4,5)

# Stage-C policy families. Coverage is learned as the largest stable positive prefix on grouped OOF calibration predictions.
ORDER_MAX_COVERAGE = 0.06 if V381_SMOKE else 0.030
MEMBERSHIP_MAX_COVERAGE = 0.08 if V381_SMOKE else 0.050
MIN_POLICY_ACTIONS_CAL = 1 if V381_SMOKE else 6
ORDER_CAL_PRECISION = 0.50 if V381_SMOKE else 0.70
MEMBERSHIP_CAL_PRECISION = 0.50 if V381_SMOKE else 0.72
ORDER_BOOT_POS = 0.0 if V381_SMOKE else 0.75
MEMBERSHIP_BOOT_POS = 0.0 if V381_SMOKE else 0.80
POLICY_BOOTSTRAPS = 30 if V381_SMOKE else 600

# Fresh promotion gates.
SAFE_MIN_MEAN = -1.0 if V381_SMOKE else 0.00050
SAFE_MIN_WORST = -1.0 if V381_SMOKE else 0.00015
SAFE_MIN_PRECISION = 0.50 if V381_SMOKE else 0.68
ATTACK_MIN_MEAN = -1.0 if V381_SMOKE else 0.00150
ATTACK_MIN_WORST = -1.0 if V381_SMOKE else 0.00055
ATTACK_MIN_PRECISION = 0.50 if V381_SMOKE else 0.65
SEALED_D_MIN_INCREMENT = -1.0 if V381_SMOKE else 0.00000
CHRONO_MIN_INCREMENT = -1.0 if V381_SMOKE else -0.00005
TEST_GEOMETRY_AUC_MAX = 1.0 if V381_SMOKE else 0.62

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE = OUT_DIR / "hf_cache_v38_1"
HF_CACHE.mkdir(parents=True, exist_ok=True)


def _seed_from_text(s):
    return int(hashlib.sha1(str(s).encode("utf-8")).hexdigest()[:8],16) % (2**31-1)

V381_WORLD_SEED = _seed_from_text("MINETODAY-V38.1-DISJOINT-AAABAC-20260830-CURRICULUM-XLMR")
SEALED_D_SALT = "MINETODAY-V36-SEALED-D-20260830-XE-COMPETENCE"

PROTECTED_TEST_USERS = {
    "U_4047", "U_4953", "U_4074", "U_4176", "U_4491", "U_4567", "U_4836", "U_4929"
}
EXPECTED_V30_CHANGED_USERS = {
    'U_4005','U_4015','U_4017','U_4043','U_4044','U_4058','U_4069','U_4070','U_4075','U_4087',
    'U_4089','U_4118','U_4121','U_4122','U_4130','U_4131','U_4137','U_4138','U_4169','U_4178',
    'U_4186','U_4214','U_4232','U_4233','U_4237','U_4265','U_4266','U_4269','U_4274','U_4285',
    'U_4300','U_4320','U_4335','U_4345','U_4354','U_4363','U_4379','U_4389','U_4394','U_4402',
    'U_4407','U_4430','U_4436','U_4438','U_4442','U_4515','U_4545','U_4555','U_4569','U_4596',
    'U_4633','U_4639','U_4661','U_4663','U_4665','U_4672','U_4687','U_4717','U_4718','U_4725',
    'U_4751','U_4759','U_4827','U_4828','U_4838','U_4886','U_4893','U_4897','U_4905','U_4919',
    'U_4933','U_4956','U_4960','U_4965','U_4978','U_4988','U_4989'
}
EXPECTED_V30_TOP1_CHANGES = 8
EXPECTED_V30_MEMBERSHIP_CHANGES = 7
EXPECTED_V31_CHANGED_VS_V30 = 60
EXPECTED_HYBRID_OVERLAP = 34
EXPECTED_STATE_COUNTS = {"S0":897, "S1":26, "S2":43, "S3":34}
EXPECTED_HYBRID_TOP5_SHA256 = "47f7db52ce8f782b8b393fd120140f1b6f51aeac452a9fabb5794cc9d8ba5e6e"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("numpy",np.__version__,"pandas",pd.__version__,"sklearn",sklearn.__version__,"lightgbm",lgb.__version__)
print("torch",torch.__version__,"transformers",transformers.__version__)
print("V381_SMOKE",V381_SMOKE,"Hybrid34",PUBLIC_HYBRID34_LB,"leader gap",round(PUBLIC_LEADER_GAP,5))
try: print("GPU runtime:\n"+subprocess.check_output(["nvidia-smi","-L"],text=True).strip())
except Exception: print("nvidia-smi unavailable; intended runtime is Kaggle T4x2.")


numpy 2.0.2 pandas 2.3.3 sklearn 1.6.1 lightgbm 4.6.0
torch 2.10.0+cu128 transformers 5.0.0
V381_SMOKE False Hybrid34 0.66006 leader gap 0.00136
GPU runtime:
GPU 0: Tesla T4 (UUID: GPU-da703750-93ff-65a8-7961-2722505cf856)
GPU 1: Tesla T4 (UUID: GPU-a3c338a5-2fd0-1b90-b88d-d1fae6e6b9eb)


In [2]:
def locate_data_dir():
    required = {
        "chat_history.csv", "modules_catalog.csv", "user_assessments.csv",
        "train_relevance.csv", "test.csv", "sample_submission.csv"
    }
    roots = [Path("/kaggle/input"), Path("/mnt/data/mine_data"), Path("/mnt/data/comp_v3"), Path("/mnt/data/comp_full"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        candidates = [root]
        if root.name == "input":
            candidates += [p for p in root.rglob("*") if p.is_dir()]
        for d in candidates:
            try:
                names = {p.name for p in d.iterdir() if p.is_file()}
            except Exception:
                continue
            if required.issubset(names):
                return d
    raise FileNotFoundError("Could not locate MineToday competition dataset directory.")

DATA_DIR = locate_data_dir()
print("DATA_DIR =", DATA_DIR)

chat = pd.read_csv(DATA_DIR / "chat_history.csv")
modules = pd.read_csv(DATA_DIR / "modules_catalog.csv")
assessments = pd.read_csv(DATA_DIR / "user_assessments.csv")
train = pd.read_csv(DATA_DIR / "train_relevance.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

MODULES = list(train.columns[1:])
N_MODULES = len(MODULES)
N_TRAIN = len(train)
N_TEST = len(test)

assert MODULES == list(sample.columns[1:])
assert N_MODULES == 17
assert N_TRAIN == 4000 and N_TEST == 1000
assert train["user_id"].is_unique and test["user_id"].is_unique

all_users = train["user_id"].tolist() + test["user_id"].tolist()
uid2idx = {u:i for i,u in enumerate(all_users)}
y = train[MODULES].to_numpy(np.float32)

print("chat      :", chat.shape)
print("modules   :", modules.shape)
print("assessment:", assessments.shape)
print("train     :", train.shape)
print("test      :", test.shape)
print("positive modules/user:", pd.Series((y > 0).sum(1)).value_counts().sort_index().to_dict())


DATA_DIR = /kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026
chat      : (14573, 4)
modules   : (17, 4)
assessment: (5000, 2)
train     : (4000, 18)
test      : (1000, 1)
positive modules/user: {4: 1362, 5: 1285, 6: 1353}


In [3]:
# Competition metric + ordered targets
DISCOUNT = 1.0 / np.log2(np.arange(2, 7))

def per_user_ndcg(y_true, scores):
    y_true=np.asarray(y_true); scores=np.asarray(scores)
    idx=np.argpartition(-scores,4,axis=1)[:,:5]
    local=np.take_along_axis(scores,idx,axis=1)
    idx=np.take_along_axis(idx,np.argsort(-local,axis=1),axis=1)
    dcg=(np.take_along_axis(y_true,idx,axis=1)*DISCOUNT).sum(1)
    ideal=np.sort(y_true,axis=1)[:,-5:][:,::-1]
    idcg=(ideal*DISCOUNT).sum(1)
    return dcg/np.maximum(idcg,1e-12)

def ndcg5(y_true,scores): return float(per_user_ndcg(y_true,scores).mean())

def top5_indices(scores):
    idx=np.argpartition(-scores,4,axis=1)[:,:5]
    local=np.take_along_axis(scores,idx,axis=1)
    return np.take_along_axis(idx,np.argsort(-local,axis=1),axis=1)

def top5_set_change_rate(a,b):
    ta,tb=top5_indices(a),top5_indices(b)
    return float(np.mean([set(x)!=set(z) for x,z in zip(ta,tb)]))

def top1_change_rate(a,b): return float(np.mean(np.argmax(a,axis=1)!=np.argmax(b,axis=1)))

def boot_ci(delta):
    delta=np.asarray(delta,float); rng=np.random.default_rng(20260823); n=len(delta); z=np.empty(BOOTSTRAPS)
    for i in range(BOOTSTRAPS): z[i]=delta[rng.integers(0,n,n)].mean()
    return tuple(float(x) for x in np.quantile(z,[.025,.975]))

Y_BIN=(y>0).astype(np.float32)
Y_TOP=np.zeros_like(y,np.float32)
Y_TOP[np.arange(N_TRAIN),np.argmax(y,axis=1)]=1.0
RANK_POS=np.full_like(y,99,dtype=np.int16)
for i,row in enumerate(y):
    p=np.flatnonzero(row>0); order=p[np.argsort(-row[p])]
    for r,m in enumerate(order,1): RANK_POS[i,m]=r
Y_TOP5=(RANK_POS<=5).astype(np.float32)

# ID order is part of the validation hypothesis, so enforce it explicitly.
def uid_num(u):
    m=re.search(r"(\d+)$",str(u))
    if m is None: raise ValueError(f"Cannot parse ordered user id: {u}")
    return int(m.group(1))
TRAIN_ID_NUM=np.asarray([uid_num(u) for u in train.user_id],np.int32)
TEST_ID_NUM=np.asarray([uid_num(u) for u in test.user_id],np.int32)
assert np.all(np.diff(TRAIN_ID_NUM)>0), "train.csv must be ordered by increasing user_id for V12"
assert np.all(np.diff(TEST_ID_NUM)>0), "test.csv must be ordered by increasing user_id for V12"
assert TEST_ID_NUM.min()>TRAIN_ID_NUM.max(), "V12 diagnostic expects test to be the next ID block"

strata=np.asarray([MODULES[j] for j in np.argmax(y,axis=1)])
ref_skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED)
ref_folds=list(ref_skf.split(np.zeros(N_TRAIN),strata))
print("train id range",TRAIN_ID_NUM.min(),TRAIN_ID_NUM.max(),"test",TEST_ID_NUM.min(),TEST_ID_NUM.max())
print("positive modules/user",pd.Series((y>0).sum(1)).value_counts().sort_index().to_dict())


train id range 1 4000 test 4001 5000
positive modules/user {4: 1362, 5: 1285, 6: 1353}


In [4]:
# Robust assessment parsing — exact V3.1 mechanism
WORD_NUM = {"nol":0, "satu":1, "dua":2, "tiga":3, "empat":4, "lima":5}
SKILL_NAMES = ["python","sql","statistics","eda","ml","ml_eval","dl","genai","business","research"]

A = np.zeros((len(all_users), 10), np.float32)
A_INVALID = np.zeros_like(A)

for row in assessments.itertuples(index=False):
    try:
        d = json.loads(row.assessment_result)
    except Exception:
        d = ast.literal_eval(row.assessment_result)

    vals, invalid = [], []
    for v in d.values():
        sv = str(v).strip().lower()
        if sv in WORD_NUM:
            value, bad = float(WORD_NUM[sv]), 0
        else:
            try:
                value = float(v)
            except Exception:
                value = np.nan
            bad = int((not np.isfinite(value)) or value < 0 or value > 5)
            if bad:
                value = np.nan
        vals.append(value)
        invalid.append(bad)

    vals = np.asarray(vals, float)
    med = float(np.nanmedian(vals)) if np.isfinite(vals).any() else 2.0
    vals = np.where(np.isfinite(vals), vals, med)
    i = uid2idx[row.user_id]
    A[i] = vals
    A_INVALID[i] = invalid

print("invalid assessment cells:", int(A_INVALID.sum()), "/", A_INVALID.size)


invalid assessment cells: 251 / 50000


In [5]:
# Exact V3.1 dataset-derived topic/style parser
TOPIC_PATTERNS = {
    "excel": r"\b(?:ms\.?\s*excel|excel|pivot(?: table)?|vlookup|hlookup|spreadsheet)\b",
    "python": r"\b(?:python|pyton|pithon|pandas(?:\s*&\s*| dan |\s+)?numpy|pandas|numpy)\b",
    "sql": r"\b(?:sql|escuel|query database|join tabel|database relasional|window function)\b",
    "statistics": r"\b(?:statistik(?:\s*&\s*probabilitas)?|probabilitas|a\s*/?\s*b testing|uji hipotesis|distribusi data)\b",
    "eda": r"\b(?:eda|exploratory data analysis|data cleaning|missing values?|matplotlib|seaborn)\b",
    "ml_eval": r"\b(?:evaluasi model(?: ml)?|metrik model|metric model|evaluasi performa model)\b",
    "mlops": r"\b(?:mlops|deployment model|deploy(?:ment)? model|ci\s*/?\s*cd|monitoring model|serving model|fastapi)\b",
    "cv": r"\b(?:computer vision|cnn|deep learning|deteksi objek|klasifikasi gambar|image classification)\b",
    "nlp": r"\b(?:natural language processing|nlp|analisis sentimen|sentiment analysis|word2vec|tf\s*-?\s*idf|transformer)\b",
    "genai": r"\b(?:generative ai|genai|large language model|llm|rag|chatgpt|chatbot|fine[- ]?tuning model bahasa)\b",
    "ml": r"\b(?:machine learning|bikin model ml|model machine learning|regresi\s*(?:&|dan)\s*klasifikasi|clustering|supervised|unsupervised|hyperparameter)\b",
    "business": r"\b(?:translate masalah bisnis ke data|menerjemahkan masalah bisnis|business(?:-| )to(?:-| )ds|permasalahan bisnis)\b",
    "dashboard": r"\b(?:dashboard|looker studio|tableau|visualisasi interaktif|storytelling data)\b",
    "webscrape": r"\b(?:web scrap(?:e|ing)|scraping|beautifulsoup|selenium|ekstraksi data.*website)\b",
    "git": r"\b(?:version control|git(?:hub|lab)?|repository|repo kode)\b",
    "prompt": r"\b(?:prompt engineering|prompting|merancang prompt|optimasi prompt)\b",
    "automation": r"\b(?:ai automation|otomasi|otomatis|automation|workflow|n8n)\b",
    "nocode": r"\b(?:no[ -]?code|zero coding|tanpa perlu (?:belajar )?(?:coding|koding)|gausah (?:coding|koding)|gak perlu (?:coding|koding)|tanpa (?:coding|koding))\b",
    "career": r"\b(?:portofolio|portfolio|cv|test interview|interview|karir|career|prospek kerja|negosiasi)\b",
}
TOPIC_RX = {k: re.compile(v, re.I) for k,v in TOPIC_PATTERNS.items()}
TOPICS = list(TOPIC_PATTERNS)

NOISE_RX = re.compile(
    r"reset password|email verifikasi|sertifikat kelulusan|pembayaran|cicilan|invoice|"
    r"akun.*(?:error|masalah)|lms.*(?:error|masalah)", re.I
)
STYLE_PATTERNS = {
    "beginner": r"pemula|dari nol|gaptek|belum pernah (?:ngoding|coding|koding)|background(?:nya)? bukan (?:it|data)|banting setir",
    "advanced": r"udah lumayan paham|sudah lumayan paham|stuck|prasyarat|syarat|\breq\b|basic apa|lanjutin",
    "urgent": r"urgent|secepatnya|sefastnya|intensif|seminggu|minggu depan",
    "budget": r"budget|diskon|worth it|cekak|fomo|kompetitor",
    "online": r"online|durasi|kapan buka|mulai minggu|jadwal",
    "nocode_style": r"tanpa perlu.*(?:coding|koding)|gausah.*(?:coding|koding)|gak perlu.*(?:coding|koding)|otomatis.*ai|solusi otomatis",
    "career_style": r"portofolio|portfolio|test interview|prospek kerja|karir|career",
}
STYLE_RX = {k: re.compile(v, re.I) for k,v in STYLE_PATTERNS.items()}
STYLES = list(STYLE_PATTERNS)

def clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"[^\w\s<>/&+-]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

def extract_topics(text):
    return [k for k,rx in TOPIC_RX.items() if rx.search(str(text))]

def make_skeleton(text):
    s = clean_text(text)
    for rx in TOPIC_RX.values():
        s = rx.sub(" <topic> ", s)
    s = re.sub(r"\b\d+\b", " <num> ", s)
    return re.sub(r"\s+", " ", s).strip()

def template_token(skeleton):
    return "tpl_" + hashlib.md5(skeleton.encode("utf-8")).hexdigest()[:10]

chat["timestamp"] = pd.to_datetime(chat["timestamp"], errors="coerce")
chat["clean"] = chat["user_chat_text"].fillna("").map(clean_text)
chat["is_noise"] = chat["user_chat_text"].fillna("").map(lambda x: bool(NOISE_RX.search(str(x))))
chat["topics"] = chat["user_chat_text"].fillna("").map(extract_topics)
chat["skeleton"] = chat["user_chat_text"].fillna("").map(make_skeleton)
for name,rx in STYLE_RX.items():
    chat["sty_"+name] = chat["user_chat_text"].fillna("").map(lambda x,r=rx: int(bool(r.search(str(x)))))

chat = chat.sort_values(["user_id","timestamp","chat_id"]).reset_index(drop=True)
user_chat = {u:g.copy() for u,g in chat.groupby("user_id", sort=False)}
print("administrative/noise rows:", int(chat["is_noise"].sum()), "/", len(chat))


administrative/noise rows: 535 / 14573


In [6]:
# V3.1 template-aware sparse representation + structured user features
USER_FEATURES = np.zeros((len(all_users), 10 + 10 + 5 + len(STYLES)*2 + len(TOPICS)*5), np.float32)

def build_user(u):
    ui = uid2idx[u]
    g = user_chat.get(u)
    if g is None:
        learn = pd.DataFrame(columns=chat.columns)
        g = learn
    else:
        learn = g[~g["is_noise"]].copy()
        if len(learn) == 0:
            learn = g.copy()

    parts = []
    n = len(learn)
    for j,row in enumerate(learn.itertuples(index=False)):
        distance = n - 1 - j
        tag = "current" if distance == 0 else ("recent" if distance <= 2 else "older")
        parts.append(f"[{tag}] {row.clean}")
        parts.append(template_token(row.skeleton))
        for tp in row.topics:
            parts.append("topic_" + tp)
            if distance == 0:
                parts.extend(["latest_topic_" + tp] * 3)
            elif distance <= 2:
                parts.extend(["recent_topic_" + tp] * 2)
        for sty in STYLES:
            if getattr(row, "sty_"+sty):
                parts.append("style_" + sty)

    for j,skill in enumerate(SKILL_NAMES):
        v = int(round(float(A[ui,j])))
        band = "low" if v <= 1 else ("mid" if v <= 3 else "high")
        parts += [f"skill_{skill}_{v}", f"skill_{skill}_bin_{band}"]

    feats = list(A[ui]) + list(A_INVALID[ui])
    total = len(g)
    learning_count = len(learn)
    noise_count = int(g["is_noise"].sum()) if total else 0
    span = 0.0
    if total >= 2 and g["timestamp"].notna().sum() >= 2:
        span = float((g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 86400.0)
    feats += [total, learning_count, noise_count, span, min(total,6)]

    for sty in STYLES:
        feats.append(float(learn["sty_"+sty].sum()) if learning_count else 0.0)
    for sty in STYLES:
        feats.append(float(learn.iloc[-1]["sty_"+sty]) if learning_count else 0.0)

    rows = list(learn.itertuples(index=False))
    nfull = len(rows)
    for tp in TOPICS:
        count = rec_count = last_pos = 0.0
        latest = 0.0
        for j,row in enumerate(rows):
            if tp in row.topics:
                count += 1.0
                rec_count += float(np.exp(-0.7 * (nfull - 1 - j)))
                last_pos = max(last_pos, (j+1) / max(nfull,1))
        if rows and tp in rows[-1].topics:
            latest = 1.0
        feats += [count, rec_count, latest, last_pos, float(count > 0)]

    USER_FEATURES[ui] = np.asarray(feats, np.float32)
    return " ".join(parts)

texts = np.asarray([build_user(u) for u in all_users], dtype=object)
assert USER_FEATURES.shape == (5000, 134)
assert np.isfinite(USER_FEATURES).all()
print("USER_FEATURES:", USER_FEATURES.shape)
print("example augmented text:", texts[1][:500])


USER_FEATURES: (5000, 134)
example augmented text: [older] min aku mau fokus jadi data analyst aja deh mau kuasain excel sama sql tpl_93c96e6b92 topic_excel topic_sql [recent] eh min gajadi deh bosku nyuruh pindah haluan bikin ai ada kelas genai nggak tpl_abc6b29bac topic_genai recent_topic_genai recent_topic_genai [recent] sori nanya mulu ya min tapi kelas yg cover data cleaning ada silabus lengkapnya ga tpl_ace230065e topic_eda recent_topic_eda recent_topic_eda [current] sori nanya mulu ya min tapi kelas belajar pyton dari nol itu berapa lama 


## Frozen public-proven backbone

The V3.1 sparse backbone, V3.1/V7 calibrator, Consensus3 profile, and V25 public-tested actions are reconstructed exactly before V31 adds any new information.


In [7]:
# Exact public-validated V3.1 sparse backbone.
t0=time.time()
word_vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.995,sublinear_tf=True,strip_accents="unicode",max_features=(2000 if V27_SMOKE else 50000),dtype=np.float32)
char_vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True,max_features=(3000 if V27_SMOKE else 60000),dtype=np.float32)
X_WORD=word_vec.fit_transform(texts)
X_CHAR=char_vec.fit_transform(texts)
X=hstack([X_WORD,X_CHAR],format="csr")
def sparse_ref(target):
    oo=np.zeros_like(y,np.float32); te=np.zeros((N_TEST,N_MODULES),np.float32)
    for tr,va in ref_folds:
        m=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000); m.fit(X[tr],target[tr]); oo[va]=m.predict(X[va]).astype(np.float32); te+=m.predict(X[N_TRAIN:]).astype(np.float32)/N_FOLDS
    return oo,te
raw_oof,raw_test=sparse_ref(y); top_oof,top_test=sparse_ref(Y_TOP)
base_oof=(.9*raw_oof+.1*top_oof).astype(np.float32); base_test=(.9*raw_test+.1*top_test).astype(np.float32)
print("X",X.shape,"base OOF",round(ndcg5(y,base_oof),6),"sec",round(time.time()-t0,1))


X (5000, 36605) base OOF 0.658254 sec 26.5


In [8]:
# Exact public-winning V3.1/V7 relevance calibration path.
def make_cal(seed,n_estimators=None):
    return lgb.LGBMClassifier(
        n_estimators=int(LGB_ESTIMATORS if n_estimators is None else n_estimators),
        learning_rate=.03,num_leaves=15,min_child_samples=50,
        reg_alpha=.5,reg_lambda=5.,colsample_bytree=.7,
        random_state=int(seed),n_jobs=4,verbosity=-1
    )

def fit_bin(seed,folds,Xtr,Xout,targets):
    oo=np.zeros((len(targets),N_MODULES),np.float32)
    out=np.zeros((Xout[0].shape[0],N_MODULES),np.float32)
    its={mid:[] for mid in MODULES}
    for m,mid in enumerate(MODULES):
        target=targets[:,m].astype(np.int32)
        for f,(tr,va) in enumerate(folds):
            clf=make_cal(seed+f+31*m)
            clf.fit(
                Xtr[m][tr],target[tr],
                eval_set=[(Xtr[m][va],target[va])],
                callbacks=[lgb.early_stopping(LGB_EARLY_STOP,verbose=False)]
            )
            it=int(clf.best_iteration_ or LGB_ESTIMATORS)
            its[mid].append(it)
            oo[va,m]=clf.predict_proba(Xtr[m][va],num_iteration=it)[:,1].astype(np.float32)
        final_iters=max(20,int(np.median(its[mid])))
        full=make_cal(seed+997+31*m,final_iters)
        full.fit(Xtr[m],target)
        out[:,m]=full.predict_proba(Xout[m])[:,1].astype(np.float32)
    return oo,out,its

META_TR=[np.column_stack([base_oof[:,m],USER_FEATURES[:N_TRAIN]]).astype(np.float32) for m in range(N_MODULES)]
META_TE=[np.column_stack([base_test[:,m],USER_FEATURES[N_TRAIN:]]).astype(np.float32) for m in range(N_MODULES)]
meta_oof,meta_test,best_iters=fit_bin(SEED,ref_folds,META_TR,META_TE,Y_BIN)
cal045_oof=(.55*base_oof+.45*meta_oof).astype(np.float32)
cal045_test=(.55*base_test+.45*meta_test).astype(np.float32)

print("sparse OOF ",round(ndcg5(y,base_oof),6))
print("meta OOF   ",round(ndcg5(y,meta_oof),6))
print("cal045 OOF ",round(ndcg5(y,cal045_oof),6),"public",PUBLIC_CAL045_LB)
print("median calibrator iterations:",{k:int(np.median(v)) for k,v in best_iters.items()})


sparse OOF  0.658254
meta OOF    0.658242
cal045 OOF  0.662646 public 0.65639
median calibrator iterations: {'M_001': 101, 'M_002': 110, 'M_003': 116, 'M_004': 40, 'M_005': 93, 'M_006': 162, 'M_007': 118, 'M_008': 80, 'M_009': 165, 'M_010': 77, 'M_011': 60, 'M_012': 96, 'M_013': 42, 'M_014': 89, 'M_015': 116, 'M_016': 91, 'M_017': 117}


In [9]:
# Exact V12.2 Consensus3 profile. No weight search in V24.
CONSENSUS3_OVERRIDES = {
    "M_002": 0.44,
    "M_006": 0.55,
    "M_007": 0.60,
    "M_010": 0.40,
    "M_011": 0.55,
    "M_012": 0.55,
    "M_015": 0.44,
    "M_017": 0.53,
}

def consensus3_vector():
    w = np.full(N_MODULES, REFERENCE_W, np.float32)
    for mid, val in CONSENSUS3_OVERRIDES.items():
        if mid not in MODULES:
            raise KeyError(mid)
        w[MODULES.index(mid)] = float(val)
    return w

CONS_W = consensus3_vector()
consensus3_oof = ((1.0 - CONS_W) * base_oof + CONS_W * meta_oof).astype(np.float32)
consensus3_test = ((1.0 - CONS_W) * base_test + CONS_W * meta_test).astype(np.float32)

ref_fold_id = np.full(N_TRAIN, -1, np.int16)
for _f, (_, _va) in enumerate(ref_folds):
    ref_fold_id[_va] = _f
assert (ref_fold_id >= 0).all()

print("Consensus3 OOF:", round(ndcg5(y, consensus3_oof), 6))
print("Consensus3 frozen weights:", {m: float(CONS_W[i]) for i,m in enumerate(MODULES)})


Consensus3 OOF: 0.664368
Consensus3 frozen weights: {'M_001': 0.44999998807907104, 'M_002': 0.4399999976158142, 'M_003': 0.44999998807907104, 'M_004': 0.44999998807907104, 'M_005': 0.44999998807907104, 'M_006': 0.550000011920929, 'M_007': 0.6000000238418579, 'M_008': 0.44999998807907104, 'M_009': 0.44999998807907104, 'M_010': 0.4000000059604645, 'M_011': 0.550000011920929, 'M_012': 0.550000011920929, 'M_013': 0.44999998807907104, 'M_014': 0.44999998807907104, 'M_015': 0.4399999976158142, 'M_016': 0.44999998807907104, 'M_017': 0.5299999713897705}


## Reconstruct the V25 protected-action package

These rows remain protected during both V30-control and V31 deployment. They are not used as supervised training labels.


In [10]:
TEST_UID_TO_ROW = {u:i for i,u in enumerate(test["user_id"].tolist())}
MID_TO_COL = {m:i for i,m in enumerate(MODULES)}

KNOWN_PB_SWAPS = [
    ("U_4047", "M_009", "M_007"),  # promote EDA over ML
    ("U_4953", "M_016", "M_005"),  # promote Version Control over MLOps
]


def swap_score_values(scores, row_idx, higher_mid, lower_mid, require_order=True):
    """Swap two numeric score values so only those two ranking positions exchange."""
    out = scores
    hi = MID_TO_COL[higher_mid]
    lo = MID_TO_COL[lower_mid]
    if require_order and not (float(out[row_idx, hi]) > float(out[row_idx, lo])):
        raise AssertionError(
            f"Expected {higher_mid}>{lower_mid} before swap, got "
            f"{out[row_idx, hi]:.8f} <= {out[row_idx, lo]:.8f}"
        )
    out[row_idx, hi], out[row_idx, lo] = out[row_idx, lo], out[row_idx, hi]
    return out

pb_test = consensus3_test.copy()
known_rows = []
for uid, higher_mid, lower_mid in KNOWN_PB_SWAPS:
    r = TEST_UID_TO_ROW[uid]
    hi, lo = MID_TO_COL[higher_mid], MID_TO_COL[lower_mid]
    margin = float(pb_test[r,hi] - pb_test[r,lo])
    known_rows.append({
        "user_id": uid, "higher_before": higher_mid, "promote": lower_mid,
        "consensus_margin": margin,
    })
    if not V30_SMOKE:
        assert 0 < margin < 0.02, f"PB reconstruction drift for {uid}: margin={margin}"
        swap_score_values(pb_test, r, higher_mid, lower_mid)
    else:
        if margin > 0:
            swap_score_values(pb_test, r, higher_mid, lower_mid, require_order=False)

known_pb_diag = pd.DataFrame(known_rows)
display(known_pb_diag)

# Exactly two users must change Top-5 ordering versus Consensus3.
cons_top5 = top5_indices(consensus3_test)
pb_top5 = top5_indices(pb_test)
changed_rows = np.flatnonzero(np.any(cons_top5 != pb_top5, axis=1))
changed_uids = test.iloc[changed_rows]["user_id"].tolist()
if not V30_SMOKE:
    assert set(changed_uids) == {"U_4047", "U_4953"}, changed_uids
print("PB reference changes:", changed_uids)


,user_id,higher_before,promote,consensus_margin
0,U_4047,M_009,M_007,0.001955
1,U_4953,M_016,M_005,0.003618


PB reference changes: ['U_4047', 'U_4953']


In [11]:
# Canonical V25 package. Notation: higher_before was above promoted before V25.
V25_CORE_ACTIONS = [
    {"user_id":"U_4074", "higher_before":"M_014", "promoted":"M_003", "expected_margin":0.000682443},
    {"user_id":"U_4176", "higher_before":"M_003", "promoted":"M_017", "expected_margin":0.000203669},
    {"user_id":"U_4491", "higher_before":"M_003", "promoted":"M_002", "expected_margin":0.001700282},
    {"user_id":"U_4567", "higher_before":"M_014", "promoted":"M_012", "expected_margin":0.000054896},
    {"user_id":"U_4836", "higher_before":"M_003", "promoted":"M_002", "expected_margin":0.000934988},
    {"user_id":"U_4929", "higher_before":"M_003", "promoted":"M_002", "expected_margin":0.000661612},
]
V25_RELAXED_ACTIONS = [
    {"user_id":"U_4030", "higher_before":"M_007", "promoted":"M_003", "expected_margin":0.002789915},
    {"user_id":"U_4650", "higher_before":"M_003", "promoted":"M_012", "expected_margin":0.002036035},
    {"user_id":"U_4731", "higher_before":"M_014", "promoted":"M_006", "expected_margin":0.001383066},
]
EXPECTED_PRE_SWAP_RANKS = {
    "U_4074": (5,6),
    "U_4176": (2,3),
    "U_4491": (4,5),
    "U_4567": (3,4),
    "U_4836": (4,5),
    "U_4929": (4,5),
}


def full_rank_matrix(scores):
    order = np.argsort(-np.asarray(scores), axis=1, kind="mergesort")
    ranks = np.empty_like(order, dtype=np.int16)
    rows = np.arange(len(order))[:,None]
    ranks[rows, order] = np.arange(1, order.shape[1]+1, dtype=np.int16)[None,:]
    return ranks


def apply_promotion(scores, action, strict_geometry=True):
    out = scores.copy()
    uid = action["user_id"]
    row = TEST_UID_TO_ROW[uid]
    hi = MID_TO_COL[action["higher_before"]]
    pr = MID_TO_COL[action["promoted"]]
    margin = float(out[row, hi] - out[row, pr])
    if strict_geometry:
        if not margin > 0:
            raise AssertionError(
                f"V25 geometry drift for {uid}: expected {action['higher_before']}>{action['promoted']} "
                f"before promotion, margin={margin:.9f}"
            )
        if abs(margin - float(action["expected_margin"])) > GEOMETRY_ATOL:
            raise AssertionError(
                f"V25 frozen-margin drift for {uid}: got {margin:.9f}, "
                f"expected {action['expected_margin']:.9f}, atol={GEOMETRY_ATOL}"
            )
    # Idempotent safety if strict geometry is disabled for debugging only.
    if margin > 0:
        out[row, hi], out[row, pr] = out[row, pr], out[row, hi]
    return out, margin

pb_ranks_before_core = full_rank_matrix(pb_test)
core_geometry_rows = []
v25_anchor_test = pb_test.copy()
for action in V25_CORE_ACTIONS:
    uid = action["user_id"]
    r = TEST_UID_TO_ROW[uid]
    hi = MID_TO_COL[action["higher_before"]]
    pr = MID_TO_COL[action["promoted"]]
    rank_pair = (int(pb_ranks_before_core[r,hi]), int(pb_ranks_before_core[r,pr]))
    expected_pair = EXPECTED_PRE_SWAP_RANKS[uid]
    if V27_STRICT_GEOMETRY:
        assert rank_pair == expected_pair, (uid, rank_pair, expected_pair)
    v25_anchor_test, margin = apply_promotion(v25_anchor_test, action, strict_geometry=V27_STRICT_GEOMETRY)
    core_geometry_rows.append({
        **action,
        "rank_higher_before": rank_pair[0],
        "rank_promoted_before": rank_pair[1],
        "current_margin": margin,
        "margin_abs_error": abs(margin-float(action["expected_margin"])),
    })

core_geometry = pd.DataFrame(core_geometry_rows)
display(core_geometry)

# V25 primary must differ from Consensus3 only on the 2 PB + 6 core users.
cons_top = top5_indices(consensus3_test)
v25_top = top5_indices(v25_anchor_test)
changed_rows_v25 = np.flatnonzero(np.any(cons_top != v25_top, axis=1))
changed_users_v25 = test.iloc[changed_rows_v25]["user_id"].tolist()
expected_v25_users = {"U_4047","U_4953", *[a["user_id"] for a in V25_CORE_ACTIONS]}
if not V30_SMOKE:
    assert set(changed_users_v25) == expected_v25_users, changed_users_v25
    assert len(changed_users_v25) == 8
print("V25 anchor reconstructed; changed users vs Consensus3:", changed_users_v25)


,user_id,higher_before,promoted,expected_margin,rank_higher_before,rank_promoted_before,current_margin,margin_abs_error
0,U_4074,M_014,M_003,0.000682,5,6,0.000682,3.803558e-10
1,U_4176,M_003,M_017,0.000204,2,3,0.000204,7.119751e-11
2,U_4491,M_003,M_002,0.001700,4,5,0.001700,9.686279e-11
3,U_4567,M_014,M_012,0.000055,3,4,0.000055,1.221619e-10
4,U_4836,M_003,M_002,0.000935,4,5,0.000935,2.602692e-10
5,U_4929,M_003,M_002,0.000662,4,5,0.000662,4.429932e-10


V25 anchor reconstructed; changed users vs Consensus3: ['U_4047', 'U_4074', 'U_4176', 'U_4491', 'U_4567', 'U_4836', 'U_4929', 'U_4953']


## V31 supervised state representation

The new block is intentionally low-dimensional and empirical. The latest learning-message intent is represented as a **topic-set signature** rather than an exact text/template identity. Candidate readiness is computed from organizer-provided assessments using the same module-to-skill map as V30, then discretized into fixed low/mid/high bands on the 0–5 scale.

Target statistics are always learned inside the relevant training partition. For outer-training rows they are generated by a second group-disjoint cross-fit, which is the critical leakage barrier for V31.


In [12]:
# Module-native lexical / intent / readiness surfaces inherited from V30 Base.
MODULE_TEXT = (
    modules.set_index("module_id").loc[MODULES, "module_name"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "description_and_syllabus"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "prerequisite_level"].astype(str)
).tolist()
MODULE_WORD = word_vec.transform(MODULE_TEXT)
MODULE_CHAR = char_vec.transform(MODULE_TEXT)
LEX_WORD = (X_WORD @ MODULE_WORD.T).toarray().astype(np.float32)
LEX_CHAR = (X_CHAR @ MODULE_CHAR.T).toarray().astype(np.float32)

MODULE_TOPIC_MAP = {
    "M_001":["excel"], "M_002":["python"], "M_003":["sql"],
    "M_004":["webscrape","python"], "M_005":["git"], "M_006":["statistics"],
    "M_007":["eda","python","statistics"], "M_008":["dashboard","sql","excel"],
    "M_009":["ml","ml_eval","eda","statistics"], "M_010":["cv","ml"],
    "M_011":["nlp","ml"], "M_012":["genai","nlp"], "M_013":["prompt","genai"],
    "M_014":["automation","genai"], "M_015":["nocode","genai"],
    "M_016":["mlops","ml","python","git"], "M_017":["career"],
}
MODULE_SKILL_MAP = {
    "M_001":[], "M_002":[0], "M_003":[1], "M_004":[0], "M_005":[9],
    "M_006":[2], "M_007":[0,2,3], "M_008":[1], "M_009":[2,3,4,5],
    "M_010":[4,6], "M_011":[4], "M_012":[4,7], "M_013":[7],
    "M_014":[7,8], "M_015":[7], "M_016":[0,4,5,9], "M_017":[8,9],
}

MODULE_COUNT = np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_RECENCY = np.zeros_like(MODULE_COUNT)
MODULE_LATEST = np.zeros_like(MODULE_COUNT)
LATEST_GROUP = np.empty(len(all_users),dtype=object); LATEST_GROUP[:] = "no_chat"
LATEST_INTENT_KEY = np.empty(len(all_users),dtype=object); LATEST_INTENT_KEY[:] = "intent:no_chat"

for u in all_users:
    ui=uid2idx[u]
    g=user_chat.get(u)
    if g is None or len(g)==0:
        continue
    learn=g[~g["is_noise"]].copy()
    if len(learn)==0:
        learn=g.copy()
    rows=list(learn.itertuples(index=False))
    if rows:
        LATEST_GROUP[ui]=rows[-1].skeleton or "empty"
        tps=sorted(set(rows[-1].topics))
        LATEST_INTENT_KEY[ui] = "intent:" + ("+".join(tps) if tps else "none")
    n=len(rows)
    for j,row in enumerate(rows):
        dist=n-1-j; w=float(np.exp(-0.7*dist)); row_topics=set(row.topics)
        for m,mid in enumerate(MODULES):
            hits=sum(tp in row_topics for tp in MODULE_TOPIC_MAP[mid])
            if hits:
                MODULE_COUNT[ui,m]+=hits
                MODULE_RECENCY[ui,m]+=w*hits
                if dist==0:
                    MODULE_LATEST[ui,m]=1.0

MODULE_READINESS=np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_INVALID=np.zeros_like(MODULE_READINESS)
for m,mid in enumerate(MODULES):
    idx=MODULE_SKILL_MAP[mid]
    if idx:
        MODULE_READINESS[:,m]=A[:,idx].mean(1)
        MODULE_INVALID[:,m]=A_INVALID[:,idx].mean(1)
    else:
        MODULE_READINESS[:,m]=A.mean(1)
        MODULE_INVALID[:,m]=A_INVALID.mean(1)

# Fixed readiness discretization consistent with the V3.1 low/mid/high assessment bins.
_ready_round=np.floor(MODULE_READINESS+0.5)
MODULE_READY_BAND=np.where(_ready_round<=1,0,np.where(_ready_round<=3,1,2)).astype(np.int8)

TRAIN_GROUP=np.asarray(LATEST_GROUP[:N_TRAIN],dtype=object)
TRAIN_INTENT=np.asarray(LATEST_INTENT_KEY[:N_TRAIN],dtype=object)
TEST_INTENT=np.asarray(LATEST_INTENT_KEY[N_TRAIN:],dtype=object)

prtxt=modules.set_index("module_id")["prerequisite_level"].astype(str).to_dict()
def module_level(mid):
    s=prtxt[mid].lower()
    if "pemula" in s: return 0
    if "menengah" in s: return 1
    if "lanjutan" in s: return 2
    if "ahli" in s: return 3
    return 4
MODULE_LEVEL=np.asarray([module_level(m) for m in MODULES],np.int16)
MODULE_LEVEL_ONEHOT=np.eye(5,dtype=np.float32)[MODULE_LEVEL]

# Ordinal LambdaRank labels: 0=irrelevant, 1=#6 ... 6=#1.
ORDINAL_GRADES=np.zeros_like(y,dtype=np.int16)
for i,row in enumerate(y):
    pos=np.flatnonzero(row>0)
    order=pos[np.argsort(-row[pos],kind="mergesort")]
    for rank,m in enumerate(order,1):
        ORDINAL_GRADES[i,m]=max(1,7-rank)
LABEL_GAIN=[0.0,0.25,0.40,0.55,0.70,0.85,1.00]

def user_strata_from_y(yy):
    yy=np.asarray(yy)
    k=(yy>0).sum(1).astype(int)
    top=np.argmax(yy,axis=1).astype(int)
    return (top*4+np.clip(k-3,0,3)).astype(int)

def make_group_folds_local(yy,groups,n_splits,seed):
    yy=np.asarray(yy); groups=np.asarray(groups,dtype=object)
    ns=min(int(n_splits),len(np.unique(groups)))
    if ns<2:
        sk=StratifiedKFold(n_splits=2,shuffle=True,random_state=int(seed))
        return list(sk.split(np.zeros(len(yy)),user_strata_from_y(yy)))
    sg=StratifiedGroupKFold(n_splits=ns,shuffle=True,random_state=int(seed))
    folds=list(sg.split(np.zeros(len(yy)),user_strata_from_y(yy),groups=groups))
    for tr,va in folds:
        assert len(set(groups[tr]).intersection(set(groups[va])))==0
    return folds

def make_group_folds_global(rows,n_splits,seed):
    rows=np.asarray(rows,int)
    local=make_group_folds_local(y[rows],TRAIN_GROUP[rows],n_splits,seed)
    return [(rows[tr],rows[va]) for tr,va in local]

print("catalog lexical matrices:",LEX_WORD.shape,LEX_CHAR.shape)
print("latest template groups:",len(np.unique(TRAIN_GROUP)))
print("latest intent signatures train/test:",len(np.unique(TRAIN_INTENT)),len(np.unique(TEST_INTENT)))
print("test intent signatures unseen in train:",len(set(TEST_INTENT)-set(TRAIN_INTENT)))
print("readiness band counts:",dict(zip(*np.unique(MODULE_READY_BAND[:N_TRAIN],return_counts=True))))
print("ordinal grade counts:",dict(zip(*np.unique(ORDINAL_GRADES,return_counts=True))))


catalog lexical matrices: (5000, 17) (5000, 17)
latest template groups: 1095
latest intent signatures train/test: 46 46
test intent signatures unseen in train: 0
readiness band counts: {np.int8(0): np.int64(45948), np.int8(1): np.int64(16376), np.int8(2): np.int64(5676)}
ordinal grade counts: {np.int16(0): np.int64(48009), np.int16(1): np.int64(1353), np.int16(2): np.int64(2638), np.int16(3): np.int64(4000), np.int16(4): np.int64(4000), np.int16(5): np.int64(4000), np.int16(6): np.int64(4000)}


In [13]:
# Strict fold-safe hierarchical Intent-State target encoding.
INTENTSTATE_KEYS = [
    "IS_PARENT_SEEN","IS_PARENT_SUPPORT_LOG","IS_PARENT_CONF",
    "IS_PARENT_POS","IS_PARENT_ORD","IS_PARENT_RAW",
    "IS_CHILD_SUPPORT_LOG","IS_CHILD_CONF",
    "IS_CHILD_POS","IS_CHILD_ORD","IS_CHILD_RAW","IS_CHILD_TOP5",
    "IS_POS_LOG_LIFT_GLOBAL","IS_ORD_DELTA_GLOBAL","IS_ORD_DELTA_PARENT","IS_RAW_DELTA_PARENT",
]

def _intent_targets(y_fit,grades_fit):
    y_fit=np.asarray(y_fit,np.float32); grades_fit=np.asarray(grades_fit,np.int16)
    return {
        "pos":(y_fit>0).astype(np.float32),
        "ord":(grades_fit.astype(np.float32)/6.0),
        "raw":y_fit.astype(np.float32),
        "top5":(grades_fit>=2).astype(np.float32),
    }

def fit_intentstate_bundle(y_fit,grades_fit,intent_fit,ready_band_fit):
    y_fit=np.asarray(y_fit,np.float32)
    grades_fit=np.asarray(grades_fit,np.int16)
    intent_fit=np.asarray(intent_fit,dtype=object)
    ready_band_fit=np.asarray(ready_band_fit,np.int8)
    n=len(y_fit)
    assert grades_fit.shape==y_fit.shape==ready_band_fit.shape
    T=_intent_targets(y_fit,grades_fit)
    glob={k:v.mean(0).astype(np.float32) for k,v in T.items()}

    parent={}
    child={}
    for key in np.unique(intent_fit):
        idx=np.flatnonzero(intent_fit==key)
        ns=int(len(idx))
        vals={}
        for name,M in T.items():
            sm=(M[idx].sum(0)+INTENT_PARENT_STRENGTH*glob[name])/(ns+INTENT_PARENT_STRENGTH)
            vals[name]=sm.astype(np.float32)
        parent[str(key)]={"n":ns,**vals}

        for m in range(N_MODULES):
            rb=ready_band_fit[idx,m]
            for b in (0,1,2):
                local=idx[rb==b]
                nc=int(len(local))
                if nc==0:
                    continue
                cv={}
                for name,M in T.items():
                    prior=float(vals[name][m])
                    cv[name]=float((M[local,m].sum()+INTENT_CHILD_STRENGTH*prior)/(nc+INTENT_CHILD_STRENGTH))
                child[(str(key),m,int(b))]={"n":nc,**cv}

    return {"n_fit":n,"global":glob,"parent":parent,"child":child}

def apply_intentstate_bundle(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object)
    ready_band=np.asarray(ready_band,np.int8)
    n=len(intent_keys)
    assert ready_band.shape==(n,N_MODULES)
    out={k:np.zeros((n,N_MODULES),np.float32) for k in INTENTSTATE_KEYS}
    glob=bundle["global"]
    logden=np.log1p(max(1,int(bundle["n_fit"])))

    for i,key0 in enumerate(intent_keys):
        key=str(key0)
        p=bundle["parent"].get(key)
        parent_seen=float(p is not None)
        if p is None:
            pn=0
            pvals={name:glob[name] for name in ("pos","ord","raw","top5")}
        else:
            pn=int(p["n"])
            pvals={name:p[name] for name in ("pos","ord","raw","top5")}
        pconf=pn/(pn+INTENT_PARENT_STRENGTH) if pn else 0.0
        pslog=np.log1p(pn)/logden if logden>0 else 0.0

        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            if c is None:
                cn=0
                cpos=float(pvals["pos"][m]); cord=float(pvals["ord"][m])
                craw=float(pvals["raw"][m]); ctop5=float(pvals["top5"][m])
            else:
                cn=int(c["n"]); cpos=float(c["pos"]); cord=float(c["ord"])
                craw=float(c["raw"]); ctop5=float(c["top5"])
            cconf=cn/(cn+INTENT_CHILD_STRENGTH) if cn else 0.0
            cslog=np.log1p(cn)/logden if logden>0 else 0.0
            gpos=float(glob["pos"][m]); gord=float(glob["ord"][m]); graw=float(glob["raw"][m])
            pord=float(pvals["ord"][m]); praw=float(pvals["raw"][m])

            out["IS_PARENT_SEEN"][i,m]=parent_seen
            out["IS_PARENT_SUPPORT_LOG"][i,m]=pslog
            out["IS_PARENT_CONF"][i,m]=pconf
            out["IS_PARENT_POS"][i,m]=float(pvals["pos"][m])
            out["IS_PARENT_ORD"][i,m]=pord
            out["IS_PARENT_RAW"][i,m]=praw
            out["IS_CHILD_SUPPORT_LOG"][i,m]=cslog
            out["IS_CHILD_CONF"][i,m]=cconf
            out["IS_CHILD_POS"][i,m]=cpos
            out["IS_CHILD_ORD"][i,m]=cord
            out["IS_CHILD_RAW"][i,m]=craw
            out["IS_CHILD_TOP5"][i,m]=ctop5
            out["IS_POS_LOG_LIFT_GLOBAL"][i,m]=float(np.clip(np.log((cpos+1e-4)/(gpos+1e-4)),-2.0,2.0))
            out["IS_ORD_DELTA_GLOBAL"][i,m]=cord-gord
            out["IS_ORD_DELTA_PARENT"][i,m]=cord-pord
            out["IS_RAW_DELTA_PARENT"][i,m]=craw-praw

    for k,v in out.items():
        if not np.isfinite(v).all():
            raise ValueError(f"non-finite Intent-State feature {k}")
    return out

def crossfit_intentstate_features(y_local,grades_local,intent_local,ready_band_local,groups_local,seed):
    y_local=np.asarray(y_local,np.float32)
    grades_local=np.asarray(grades_local,np.int16)
    intent_local=np.asarray(intent_local,dtype=object)
    ready_band_local=np.asarray(ready_band_local,np.int8)
    groups_local=np.asarray(groups_local,dtype=object)
    out={k:np.zeros_like(y_local,np.float32) for k in INTENTSTATE_KEYS}
    folds=make_group_folds_local(y_local,groups_local,INTENT_INNER_FOLDS,seed)
    seen=np.zeros(len(y_local),bool)
    for tr,va in folds:
        assert len(set(groups_local[tr]).intersection(set(groups_local[va])))==0
        bundle=fit_intentstate_bundle(y_local[tr],grades_local[tr],intent_local[tr],ready_band_local[tr])
        vv=apply_intentstate_bundle(bundle,intent_local[va],ready_band_local[va])
        for k in INTENTSTATE_KEYS:
            out[k][va]=vv[k]
        seen[va]=True
    assert seen.all(), "Intent-State crossfit did not cover every training row"
    return out

def intentstate_support_audit(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object); ready_band=np.asarray(ready_band,np.int8)
    supports=[]; parent_support=[]; parent_seen=[]
    for i,key0 in enumerate(intent_keys):
        key=str(key0); p=bundle["parent"].get(key)
        pn=int(p["n"]) if p is not None else 0
        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            supports.append(int(c["n"]) if c is not None else 0)
            parent_support.append(pn); parent_seen.append(int(p is not None))
    s=np.asarray(supports,int); ps=np.asarray(parent_support,int); pseen=np.asarray(parent_seen,int)
    rows=[]
    for thr in [1,5,10,20,30,50]:
        rows.append({"level":"child","threshold":thr,"coverage":float(np.mean(s>=thr))})
    rows.append({"level":"child","threshold":-1,"coverage":1.0,"median_support":float(np.median(s)),"p10_support":float(np.quantile(s,.10)),"p25_support":float(np.quantile(s,.25))})
    rows.append({"level":"parent","threshold":1,"coverage":float(np.mean(pseen>0)),"median_support":float(np.median(ps))})
    return pd.DataFrame(rows)

print("strict nested Intent-State TE helpers ready; features:",len(INTENTSTATE_KEYS))


strict nested Intent-State TE helpers ready; features: 16


## Exact V30 Base ranker + V31 feature branch

The Base portion below preserves the V30 long-format feature order exactly. Both control and candidate use the same LambdaRank hyperparameters, same outer folds, same anchor, same `alpha`, and the same ranker seed inside each fold. The only experimental difference is whether the Intent-State columns are present.


## Hybrid34 state machine and frozen ranking surfaces

The public result defines four states from `V25 → V30 → V31`: S0 stable/stable, S1 stable then V31-change (public-toxic), S2 V30-change then V31-abstain, and S3 V30-change plus V31-change (the 34-row public-positive region). V38 uses the same exact state machine. S1 is never touched and S3 is immutable.


In [14]:
def rank_matrix(scores):
    order=np.argsort(-np.asarray(scores),axis=1,kind="mergesort")
    ranks=np.empty_like(order,dtype=np.int16)
    ranks[np.arange(len(order))[:,None],order]=np.arange(1,N_MODULES+1,dtype=np.int16)[None,:]
    return ranks

def build_long_matrix_v31(rows_global,anchor,base,meta,intentstate=None):
    rows_global=np.asarray(rows_global,int); anchor=np.asarray(anchor,np.float32)
    base=np.asarray(base,np.float32); meta=np.asarray(meta,np.float32); n=len(rows_global)
    ranks=rank_matrix(anchor).astype(np.float32)
    top1=anchor.max(1,keepdims=True); sorted_scores=np.sort(anchor,axis=1)[:,::-1]; cut5=sorted_scores[:,4:5]
    mu=anchor.mean(1,keepdims=True); sd=np.maximum(anchor.std(1,keepdims=True),1e-4); z=(anchor-mu)/sd
    order=np.argsort(-anchor,axis=1,kind="mergesort"); prev_gap=np.zeros_like(anchor); next_gap=np.zeros_like(anchor)
    for i in range(n):
        oo=order[i]; ss=anchor[i,oo]
        pg=np.r_[0.0,ss[:-1]-ss[1:]]; ng=np.r_[ss[:-1]-ss[1:],0.0]
        prev_gap[i,oo]=pg; next_gap[i,oo]=ng
    base_mats=[anchor,ranks/float(N_MODULES),(ranks<=5).astype(np.float32),top1-anchor,anchor-cut5,
               prev_gap,next_gap,z,base,meta,meta-base,LEX_WORD[rows_global],LEX_CHAR[rows_global],
               MODULE_COUNT[rows_global],MODULE_RECENCY[rows_global],MODULE_LATEST[rows_global],
               MODULE_READINESS[rows_global],MODULE_INVALID[rows_global]]
    base_names=["anchor_score","anchor_rank_norm","anchor_top5","top1_minus_score","score_minus_cut5",
                "prev_gap","next_gap","anchor_z","base_score","meta_score","meta_minus_base",
                "lex_word","lex_char","topic_count","topic_recency","topic_latest","readiness","invalid_readiness"]
    pieces=[np.column_stack([m.reshape(-1) for m in base_mats]).astype(np.float32)]; names=list(base_names)
    pieces.append(np.repeat(USER_FEATURES[rows_global],N_MODULES,axis=0).astype(np.float32)); names += [f"user_f{i:03d}" for i in range(USER_FEATURES.shape[1])]
    pieces.append(np.tile(np.eye(N_MODULES,dtype=np.float32),(n,1))); names += [f"module_{m}" for m in MODULES]
    pieces.append(np.tile(MODULE_LEVEL_ONEHOT,(n,1)).astype(np.float32)); names += [f"module_level_{i}" for i in range(5)]
    bounds={"base":sum(p.shape[1] for p in pieces)}
    if intentstate is not None:
        mats=[intentstate[k] for k in INTENTSTATE_KEYS]
        pieces.append(np.column_stack([m.reshape(-1) for m in mats]).astype(np.float32)); names += [k.lower() for k in INTENTSTATE_KEYS]
        bounds["intentstate"]=sum(p.shape[1] for p in pieces)
    XX=np.hstack(pieces).astype(np.float32,copy=False)
    assert XX.shape[0]==n*N_MODULES and len(names)==XX.shape[1] and np.isfinite(XX).all()
    return XX,names,bounds

def make_lambdarank(seed):
    return lgb.LGBMRanker(objective="lambdarank",metric="ndcg",n_estimators=RANKER_ESTIMATORS,
        learning_rate=RANKER_LEARNING_RATE,num_leaves=15,min_child_samples=80,reg_alpha=1.0,reg_lambda=10.0,
        colsample_bytree=0.78,subsample=0.90,subsample_freq=1,max_bin=127,lambdarank_truncation_level=8,
        label_gain=LABEL_GAIN,random_state=int(seed),n_jobs=4,verbosity=-1,force_col_wise=True)

def fit_ranker_predict(kind,Xtr,Xva,grades_tr,n_users,seed):
    if kind!="lambda": raise ValueError("V38 intentionally supports only the proven LambdaRank V30/V31 backbone.")
    model=make_lambdarank(seed); group=np.full(int(n_users),N_MODULES,dtype=np.int32)
    model.fit(Xtr,grades_tr.reshape(-1).astype(np.int32),group=group)
    pred=model.predict(Xva).reshape(-1,N_MODULES).astype(np.float32)
    return model,pred

def align_ranker_to_anchor(raw_ranker,anchor):
    raw=np.asarray(raw_ranker,np.float32); anchor=np.asarray(anchor,np.float32)
    rmu=raw.mean(1,keepdims=True); rsd=np.maximum(raw.std(1,keepdims=True),1e-5)
    amu=anchor.mean(1,keepdims=True); asd=np.maximum(anchor.std(1,keepdims=True),1e-5)
    return (amu+(raw-rmu)*(asd/rsd)).astype(np.float32)

def blend_residual(anchor,raw_ranker,alpha=ALPHA):
    aligned=align_ranker_to_anchor(raw_ranker,anchor)
    return (anchor+float(alpha)*(aligned-anchor)).astype(np.float32)

def state_codes(pre_anchor,v30,v31):
    pre_top=top5_indices(pre_anchor); v30_top=top5_indices(v30); v31_top=top5_indices(v31)
    a=np.any(pre_top!=v30_top,axis=1); b=np.any(v30_top!=v31_top,axis=1)
    return (2*a.astype(np.int8)+b.astype(np.int8)).astype(np.int8)

def hybrid34_surface(pre_anchor,v30,v31):
    st=state_codes(pre_anchor,v30,v31); out=np.asarray(v30,np.float32).copy(); out[st==3]=np.asarray(v31,np.float32)[st==3]
    return out,st

def cluster_boot_ci(values,groups,seed,B=BOOTSTRAPS):
    values=np.asarray(values,float); groups=np.asarray(groups,dtype=object); ug=np.unique(groups)
    by={g:np.flatnonzero(groups==g) for g in ug}; rng=np.random.default_rng(int(seed)); z=np.empty(int(B),float)
    for b in range(int(B)):
        sampled=rng.choice(ug,size=len(ug),replace=True); idx=np.concatenate([by[g] for g in sampled]); z[b]=values[idx].mean()
    return tuple(float(x) for x in np.quantile(z,[.025,.975]))


## Fold-pure Hybrid34 reference helper

Every XLM-R training example is generated from an **honest Hybrid34 analogue**. Training-side pair candidates are produced by an inner cross-fit; calibration and outer validation candidates are produced by models that never see the corresponding user labels. No XE/Yeti ranker is trained or queried anywhere in V38.


In [15]:
def _make_cal_local(seed,n_estimators=None):
    return lgb.LGBMClassifier(n_estimators=int(LGB_ESTIMATORS if n_estimators is None else n_estimators),
        learning_rate=.03,num_leaves=15,min_child_samples=50,reg_alpha=.5,reg_lambda=5.,colsample_bytree=.7,
        random_state=int(seed),n_jobs=4,verbosity=-1)

def build_anchor_split_full(train_rows,val_rows,seed):
    train_rows=np.asarray(train_rows,int); val_rows=np.asarray(val_rows,int)
    local_y=y[train_rows]; local_top=Y_TOP[train_rows]; local_bin=Y_BIN[train_rows]
    folds=make_group_folds_local(local_y,TRAIN_GROUP[train_rows],ANCHOR_INNER_FOLDS,seed); n=len(train_rows)
    raw=np.zeros((n,N_MODULES),np.float32); top=np.zeros_like(raw)
    for lt,lv in folds:
        mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000); mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mr.fit(X[train_rows[lt]],local_y[lt]); mt.fit(X[train_rows[lt]],local_top[lt])
        raw[lv]=mr.predict(X[train_rows[lv]]).astype(np.float32); top[lv]=mt.predict(X[train_rows[lv]]).astype(np.float32)
    mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000); mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mr.fit(X[train_rows],local_y); mt.fit(X[train_rows],local_top)
    base_o=(.9*raw+.1*top).astype(np.float32); base_v=(.9*mr.predict(X[val_rows])+.1*mt.predict(X[val_rows])).astype(np.float32)
    meta_o=np.zeros_like(base_o); meta_v=np.zeros((len(val_rows),N_MODULES),np.float32)
    for m in range(N_MODULES):
        xt=np.column_stack([base_o[:,m],USER_FEATURES[train_rows]]).astype(np.float32); xv=np.column_stack([base_v[:,m],USER_FEATURES[val_rows]]).astype(np.float32)
        yy=local_bin[:,m].astype(np.int32); its=[]
        for f,(lt,lv) in enumerate(folds):
            clf=_make_cal_local(seed+31*m+f); clf.fit(xt[lt],yy[lt],eval_set=[(xt[lv],yy[lv])],callbacks=[lgb.early_stopping(LGB_EARLY_STOP,verbose=False)])
            it=int(clf.best_iteration_ or LGB_ESTIMATORS); its.append(it); meta_o[lv,m]=clf.predict_proba(xt[lv],num_iteration=it)[:,1].astype(np.float32)
        final_it=max(20,int(np.median(its))); full=_make_cal_local(seed+997+31*m,final_it); full.fit(xt,yy)
        meta_v[:,m]=full.predict_proba(xv)[:,1].astype(np.float32)
    cons_o=((1.0-CONS_W)*base_o+CONS_W*meta_o).astype(np.float32); cons_v=((1.0-CONS_W)*base_v+CONS_W*meta_v).astype(np.float32)
    return {"base_train":base_o,"meta_train":meta_o,"cons_train":cons_o,"base_val":base_v,"meta_val":meta_v,"cons_val":cons_v}

def fit_reference_hybrid(train_rows,val_rows,seed):
    train_rows=np.asarray(train_rows,int); val_rows=np.asarray(val_rows,int)
    anc=build_anchor_split_full(train_rows,val_rows,seed+701)
    intent_tr=crossfit_intentstate_features(y[train_rows],ORDINAL_GRADES[train_rows],TRAIN_INTENT[train_rows],MODULE_READY_BAND[train_rows],TRAIN_GROUP[train_rows],seed+809)
    bundle=fit_intentstate_bundle(y[train_rows],ORDINAL_GRADES[train_rows],TRAIN_INTENT[train_rows],MODULE_READY_BAND[train_rows])
    intent_va=apply_intentstate_bundle(bundle,TRAIN_INTENT[val_rows],MODULE_READY_BAND[val_rows])
    Xtr,_,bounds=build_long_matrix_v31(train_rows,anc["cons_train"],anc["base_train"],anc["meta_train"],intent_tr)
    Xva,_,_=build_long_matrix_v31(val_rows,anc["cons_val"],anc["base_val"],anc["meta_val"],intent_va)
    rank_seed=seed+907
    m0,r0=fit_ranker_predict("lambda",Xtr[:,:bounds["base"]],Xva[:,:bounds["base"]],ORDINAL_GRADES[train_rows],len(train_rows),rank_seed)
    m1,r1=fit_ranker_predict("lambda",Xtr[:,:bounds["intentstate"]],Xva[:,:bounds["intentstate"]],ORDINAL_GRADES[train_rows],len(train_rows),rank_seed)
    v30=blend_residual(anc["cons_val"],r0,ALPHA); v31=blend_residual(anc["cons_val"],r1,ALPHA); hybrid,st=hybrid34_surface(anc["cons_val"],v30,v31)
    out={"pre":anc["cons_val"].astype(np.float32),"v30":v30,"v31":v31,"hybrid":hybrid,"state":st}
    del Xtr,Xva,intent_tr,intent_va,bundle,m0,m1,anc; gc.collect(); return out

def crossfit_hybrid_rows(rows,seed,tag):
    rows=np.asarray(rows,int); key=hashlib.sha1(np.asarray(rows,np.int32).tobytes()+str(seed).encode()).hexdigest()[:12]
    cache=OUT_DIR/f"v38_hybrid_oof_{re.sub(r'[^A-Za-z0-9_-]+','_',tag)}_{key}.npz"
    if cache.exists():
        z=np.load(cache); return {k:z[k] for k in ("pre","v30","v31","hybrid","state")}
    out={"pre":np.zeros((len(rows),N_MODULES),np.float32),"v30":np.zeros((len(rows),N_MODULES),np.float32),
         "v31":np.zeros((len(rows),N_MODULES),np.float32),"hybrid":np.zeros((len(rows),N_MODULES),np.float32),"state":np.zeros(len(rows),np.int8)}
    pos={r:i for i,r in enumerate(rows.tolist())}
    folds=make_group_folds_global(rows,INNER_HYBRID_FOLDS,seed)
    for f,(tr,va) in enumerate(folds):
        b=fit_reference_hybrid(tr,va,seed+1009*f)
        loc=np.asarray([pos[int(r)] for r in va],int)
        for k in out: out[k][loc]=b[k]
        del b; gc.collect()
    np.savez_compressed(cache,**out)
    return out

print("Fold-pure Hybrid34 builders ready. XLM-R branch never calls XE/Yeti.")


Fold-pure Hybrid34 builders ready. XLM-R branch never calls XE/Yeti.


## Stage A — broad organizer-only user↔module semantic adaptation

The V38 failure showed that rare Hybrid34 residual labels are too difficult as the *first* task-specific signal. Stage A therefore teaches XLM-R the broad MineToday task before any residual calibration. Each fit-side user contributes all 17 organizer modules. The model jointly learns binary relevance (`score > 0`) and the organizer relevance score in `[0,1]`. No Hybrid34 score, module rank, or handcrafted anchor feature enters Stage A.


In [16]:
def locate_xlmr_source():
    env=os.getenv("V381_XLMR_MODEL_PATH","").strip()
    if env and Path(env).exists(): return env
    root=Path("/kaggle/input")
    if root.exists():
        hits=[]
        for p in root.rglob("config.json"):
            try:
                cfg=json.load(open(p)); mt=str(cfg.get("model_type","")).lower(); hs=int(cfg.get("hidden_size",0)); nl=int(cfg.get("num_hidden_layers",0))
                if mt in {"xlm-roberta","xlm_roberta"} and hs==768 and nl==12: hits.append(str(p.parent))
            except Exception: pass
        if hits: return sorted(hits)[0]
    return XLMR_MODEL_ID

XLMR_SOURCE=locate_xlmr_source()
print("XLM-R source:",XLMR_SOURCE)
TOKENIZER=AutoTokenizer.from_pretrained(XLMR_SOURCE,cache_dir=str(HF_CACHE),use_fast=True)

mod_idx=modules.set_index("module_id").loc[MODULES]
MODULE_DOCS=[f"{row.module_name}. Materi: {row.description_and_syllabus}. Prasyarat: {row.prerequisite_level}." for _,row in mod_idx.iterrows()]

def build_v381_context(user_id):
    g=user_chat.get(user_id)
    if g is None or len(g)==0: return "[NO CHAT HISTORY]"
    learn=g[~g["is_noise"]].copy()
    if len(learn)==0: learn=g.copy()
    msgs=[str(x).strip() for x in learn["clean"].tolist() if str(x).strip()]
    if not msgs: return "[NO CHAT HISTORY]"
    rev=msgs[::-1]; parts=[f"[CURRENT] {rev[0]}"]
    for i,m in enumerate(rev[1:3],1): parts.append(f"[RECENT {i}] {m}")
    if len(rev)>3: parts.append("[OLDER] "+" ".join(rev[3:6]))
    return " ".join(parts)

V381_CONTEXT=np.asarray([build_v381_context(u) for u in all_users],dtype=object)
V381_CONTEXT_LEN=np.asarray([len(TOKENIZER.encode(x,add_special_tokens=False)) for x in V381_CONTEXT],int)
TOKENIZER_AUDIT=pd.DataFrame([{
    "model_source":str(XLMR_SOURCE),"max_length":XLMR_MAX_LENGTH,"median_context_tokens":float(np.median(V381_CONTEXT_LEN)),
    "p90_context_tokens":float(np.quantile(V381_CONTEXT_LEN,.90)),"p95_context_tokens":float(np.quantile(V381_CONTEXT_LEN,.95)),
    "max_context_tokens":int(V381_CONTEXT_LEN.max()),"contexts_over_192":int(np.sum(V381_CONTEXT_LEN>192)),
    "contexts_over_224":int(np.sum(V381_CONTEXT_LEN>224)),"no_chat_contexts":int(np.sum(V381_CONTEXT=="[NO CHAT HISTORY]"))
}])
display(TOKENIZER_AUDIT)

class RelevanceGridDataset(Dataset):
    def __init__(self,rows_global,with_labels=False):
        self.rows=np.asarray(rows_global,int); self.with_labels=bool(with_labels)
        ctx=[]; docs=[]; rg=[]; mm=[]
        for r in self.rows:
            for m in range(N_MODULES):
                ctx.append(str(V381_CONTEXT[int(r)])); docs.append(MODULE_DOCS[m]); rg.append(int(r)); mm.append(m)
        self.row_global=np.asarray(rg,int); self.module=np.asarray(mm,int)
        self.tok=TOKENIZER(ctx,docs,padding="max_length",truncation="longest_first",max_length=XLMR_MAX_LENGTH,return_tensors="pt")
        if self.with_labels:
            yy=y[self.row_global,self.module].astype(np.float32)
            self.raw=torch.tensor(yy,dtype=torch.float32); self.rel=torch.tensor((yy>0).astype(np.float32),dtype=torch.float32)
        else: self.raw=self.rel=None
    def __len__(self): return len(self.module)
    def __getitem__(self,i):
        z={k:v[i] for k,v in self.tok.items()}
        if self.raw is not None: z.update(label_rel=self.rel[i],label_raw=self.raw[i])
        return z

class XLMRRelevanceModel(nn.Module):
    def __init__(self,source):
        super().__init__(); self.encoder=AutoModel.from_pretrained(source,cache_dir=str(HF_CACHE)); h=int(self.encoder.config.hidden_size)
        self.drop=nn.Dropout(.12); self.rel_head=nn.Linear(h,1); self.score_head=nn.Linear(h,1)
    def forward(self,input_ids,attention_mask,**kwargs):
        o=self.encoder(input_ids=input_ids,attention_mask=attention_mask); h=self.drop(o.last_hidden_state[:,0])
        return self.rel_head(h).squeeze(-1),torch.sigmoid(self.score_head(h).squeeze(-1))

def set_all_seeds(seed):
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))

def _core_model(model): return model.module if isinstance(model,nn.DataParallel) else model

def _stagea_predict(model,ds,batch_size=None):
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.eval(); lp=[]; sp=[]
    loader=DataLoader(ds,batch_size=int(batch_size or STAGEA_BATCH),shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    with torch.no_grad():
        for batch in loader:
            inp={k:v.to(device,non_blocking=True) for k,v in batch.items() if k in ("input_ids","attention_mask")}
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=torch.cuda.is_available()): lo,ss=model(**inp)
            lp.append(torch.sigmoid(lo).detach().float().cpu().numpy()); sp.append(ss.detach().float().cpu().numpy())
    return np.concatenate(lp),np.concatenate(sp)

def train_stagea(train_ds,seed,tag):
    set_all_seeds(seed); device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model=XLMRRelevanceModel(XLMR_SOURCE).to(device)
    if torch.cuda.device_count()>=2: model=nn.DataParallel(model)
    core=_core_model(model)
    rel=np.asarray(train_ds.rel.numpy(),float); pos=max(float(rel.sum()),1.0); neg=max(float(len(rel)-rel.sum()),1.0); pos_weight=torch.tensor(neg/pos,dtype=torch.float32,device=device)
    enc=list(core.encoder.parameters()); heads=list(core.rel_head.parameters())+list(core.score_head.parameters())
    opt=torch.optim.AdamW([{"params":enc,"lr":STAGEA_ENCODER_LR},{"params":heads,"lr":STAGEA_HEAD_LR}],weight_decay=XLMR_WEIGHT_DECAY)
    gen=torch.Generator(); gen.manual_seed(int(seed)); loader=DataLoader(train_ds,batch_size=STAGEA_BATCH,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available(),generator=gen)
    steps=max(1,math.ceil(len(loader)/STAGEA_GRAD_ACCUM)*STAGEA_EPOCHS); warm=int(steps*XLMR_WARMUP); sched=get_linear_schedule_with_warmup(opt,warm,steps)
    scaler=torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available()); history=[]
    for ep in range(STAGEA_EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); losses=[]
        for bi,batch in enumerate(loader):
            inp={k:v.to(device,non_blocking=True) for k,v in batch.items() if k in ("input_ids","attention_mask")}; yr=batch["label_rel"].to(device); ys=batch["label_raw"].to(device)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=torch.cuda.is_available()):
                lo,ss=model(**inp); bce=nn.functional.binary_cross_entropy_with_logits(lo,yr,pos_weight=pos_weight); reg=nn.functional.smooth_l1_loss(ss,ys); loss=(bce+STAGEA_SCORE_WEIGHT*reg)/STAGEA_GRAD_ACCUM
            scaler.scale(loss).backward(); losses.append(float(loss.detach().cpu())*STAGEA_GRAD_ACCUM)
            if (bi+1)%STAGEA_GRAD_ACCUM==0 or bi+1==len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),XLMR_GRAD_CLIP); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        history.append({"stage":"A","tag":str(tag),"seed":int(seed),"epoch":ep+1,"train_loss":float(np.mean(losses))})
    return model,history

def stagea_matrix_from_dataset(model,ds,n_users):
    pr,sc=_stagea_predict(model,ds); return pr.reshape(n_users,N_MODULES),sc.reshape(n_users,N_MODULES)

print("Stage A broad relevance curriculum ready")


XLM-R source: FacebookAI/xlm-roberta-base


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

,model_source,max_length,median_context_tokens,p90_context_tokens,p95_context_tokens,max_context_tokens,contexts_over_192,contexts_over_224,no_chat_contexts
0,FacebookAI/xlm-roberta-base,256,61.0,131.0,143.0,199,3,0,150


Stage A broad relevance curriculum ready


## Stage B — hard non-tie pairwise semantic curriculum

Stage B starts from the Stage-A encoder rather than the generic pretrained checkpoint. Pair supervision is built only from fit-side organizer labels and an honest fit-side Hybrid34 OOF surface. We include true adjacent positive preferences, non-tie Hybrid top-6 adjacent pairs, and low-positive vs high-ranked hard-negative comparisons. Pair reversal is exact. The lower six XLM-R layers and embeddings remain frozen; only the upper six transformer layers + a tiny directional head adapt.


In [17]:
def build_hard_pair_table(rows_global,hybrid_oof):
    rows_global=np.asarray(rows_global,int); hybrid_oof=np.asarray(hybrid_oof,np.float32); store={}
    def add(rg,a,b,source,priority):
        if int(a)==int(b): return
        ya=float(y[int(rg),int(a)]); yb=float(y[int(rg),int(b)])
        if abs(ya-yb)<=1e-12: return
        aa,bb=sorted((int(a),int(b))); key=(int(rg),aa,bb); diff=abs(float(y[int(rg),bb]-y[int(rg),aa])); weight=1.0+2.0*min(diff,1.0)+0.15*float(priority)
        rec={"row_global":int(rg),"user_id":all_users[int(rg)],"a":aa,"b":bb,"label":int(y[int(rg),bb]>y[int(rg),aa]),"weight":float(weight),"source":str(source),"delta_y":float(y[int(rg),bb]-y[int(rg),aa])}
        if key not in store or weight>store[key]["weight"]: store[key]=rec
    for local,rg in enumerate(rows_global):
        gt=y[int(rg)]; sc=hybrid_oof[local]
        pos=np.flatnonzero(gt>0); pos=pos[np.argsort(-gt[pos],kind="mergesort")]
        for j in range(max(0,len(pos)-1)): add(rg,pos[j],pos[j+1],"true_adjacent_positive",3)
        ho=np.argsort(-sc,kind="mergesort")[:6]
        for j in range(5): add(rg,ho[j],ho[j+1],"hybrid_top6_nontie",4)
        irr=[m for m in np.argsort(-sc,kind="mergesort") if gt[m]<=0][:2]
        low_pos=list(pos[-min(2,len(pos)):])
        for p in low_pos:
            for q in irr: add(rg,p,q,"low_positive_vs_hard_negative",5)
    out=pd.DataFrame(list(store.values()))
    if len(out): out=out.sort_values(["row_global","a","b"],kind="mergesort").reset_index(drop=True)
    return out

def _pair_text(tab,reverse=False):
    ctx=[]; docs=[]
    for r in tab.itertuples(index=False):
        a=int(r.b if reverse else r.a); b=int(r.a if reverse else r.b)
        ctx.append(str(V381_CONTEXT[int(r.row_global)])); docs.append(f"Candidate A: {MODULE_DOCS[a]} Candidate B: {MODULE_DOCS[b]}")
    return ctx,docs

class PairPreferenceDataset(Dataset):
    def __init__(self,tab,reverse=False,with_labels=True):
        self.tab=tab.reset_index(drop=True); self.reverse=bool(reverse); self.with_labels=bool(with_labels)
        ctx,docs=_pair_text(self.tab,reverse=reverse); self.tok=TOKENIZER(ctx,docs,padding="max_length",truncation="longest_first",max_length=XLMR_MAX_LENGTH,return_tensors="pt")
        if self.with_labels:
            lab=self.tab.label.to_numpy(np.float32); wt=self.tab.weight.to_numpy(np.float32) if "weight" in self.tab else np.ones(len(self.tab),np.float32)
            if reverse: lab=1.0-lab
            self.lab=torch.tensor(lab,dtype=torch.float32); self.wt=torch.tensor(wt,dtype=torch.float32)
        else: self.lab=self.wt=None
    def __len__(self): return len(self.tab)
    def __getitem__(self,i):
        z={k:v[i] for k,v in self.tok.items()}
        if self.lab is not None: z.update(label=self.lab[i],sample_weight=self.wt[i])
        return z

class XLMRPairPreference(nn.Module):
    def __init__(self,source,encoder_state):
        super().__init__(); self.encoder=AutoModel.from_pretrained(source,cache_dir=str(HF_CACHE)); self.encoder.load_state_dict(encoder_state,strict=True); h=int(self.encoder.config.hidden_size); self.drop=nn.Dropout(.12); self.head=nn.Linear(h,1)
        for p in self.encoder.parameters(): p.requires_grad=False
        layers=list(self.encoder.encoder.layer); assert len(layers)>=STAGEB_TOP_LAYERS
        for layer in layers[-STAGEB_TOP_LAYERS:]:
            for p in layer.parameters(): p.requires_grad=True
    def forward(self,input_ids,attention_mask,**kwargs):
        o=self.encoder(input_ids=input_ids,attention_mask=attention_mask); return self.head(self.drop(o.last_hidden_state[:,0])).squeeze(-1)

def _stageb_predict(model,ds,batch_size=None):
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.eval(); out=[]; loader=DataLoader(ds,batch_size=int(batch_size or STAGEB_BATCH),shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    with torch.no_grad():
        for batch in loader:
            inp={k:v.to(device,non_blocking=True) for k,v in batch.items() if k in ("input_ids","attention_mask")}
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=torch.cuda.is_available()): lo=model(**inp)
            out.append(torch.sigmoid(lo).detach().float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0,float)

def train_stageb(train_f,train_r,encoder_state,seed,tag):
    set_all_seeds(seed); device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model=XLMRPairPreference(XLMR_SOURCE,encoder_state).to(device)
    if torch.cuda.device_count()>=2: model=nn.DataParallel(model)
    core=_core_model(model); enc=[p for p in core.encoder.parameters() if p.requires_grad]; head=list(core.head.parameters())
    opt=torch.optim.AdamW([{"params":enc,"lr":STAGEB_ENCODER_LR},{"params":head,"lr":STAGEB_HEAD_LR}],weight_decay=XLMR_WEIGHT_DECAY)
    class _Both(Dataset):
        def __len__(self): return len(train_f)+len(train_r)
        def __getitem__(self,i): return train_f[i] if i<len(train_f) else train_r[i-len(train_f)]
    ds=_Both(); gen=torch.Generator(); gen.manual_seed(int(seed)); loader=DataLoader(ds,batch_size=STAGEB_BATCH,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available(),generator=gen)
    steps=max(1,math.ceil(len(loader)/STAGEB_GRAD_ACCUM)*STAGEB_EPOCHS); warm=int(steps*XLMR_WARMUP); sched=get_linear_schedule_with_warmup(opt,warm,steps); scaler=torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available()); history=[]
    for ep in range(STAGEB_EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); losses=[]
        for bi,batch in enumerate(loader):
            inp={k:v.to(device,non_blocking=True) for k,v in batch.items() if k in ("input_ids","attention_mask")}; yy=batch["label"].to(device); ww=batch["sample_weight"].to(device)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=torch.cuda.is_available()): lo=model(**inp); loss=(nn.functional.binary_cross_entropy_with_logits(lo,yy,reduction="none")*ww).mean()/STAGEB_GRAD_ACCUM
            scaler.scale(loss).backward(); losses.append(float(loss.detach().cpu())*STAGEB_GRAD_ACCUM)
            if (bi+1)%STAGEB_GRAD_ACCUM==0 or bi+1==len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],XLMR_GRAD_CLIP); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        history.append({"stage":"B","tag":str(tag),"seed":int(seed),"epoch":ep+1,"train_loss":float(np.mean(losses)),"trainable_params":int(sum(p.numel() for p in model.parameters() if p.requires_grad))})
    return model,history

print("Stage B hard pair curriculum ready")


Stage B hard pair curriculum ready


## Stage C — frozen semantic residual calibration

The rare Hybrid34 residual labels are **not** allowed to update XLM-R weights. Two independently trained semantic curricula produce broad relevance scores and directional pair preferences. These are reduced to a small semantic/geometry vector and passed to grouped cross-fitted Logistic/Ridge calibrators. Policy size is learned as a calibration-side coverage prefix, not by arbitrary neural probability thresholds.


In [18]:
def build_residual_table(rows_global,hybrid,state,with_labels=True):
    rows_global=np.asarray(rows_global,int); hybrid=np.asarray(hybrid,np.float32); state=np.asarray(state,np.int8); rows=[]
    for local,(rg,sc,st) in enumerate(zip(rows_global,hybrid,state)):
        if int(st)!=0: continue
        order=np.argsort(-sc,kind="mergesort")
        for pos in ELIGIBLE_POSITIONS:
            a=int(order[pos-1]); b=int(order[pos]); rec={"row_local":int(local),"row_global":int(rg),"user_id":all_users[int(rg)],"position":int(pos),"membership":int(pos==5),"a":a,"b":b,"module_a":MODULES[a],"module_b":MODULES[b],"margin":float(sc[a]-sc[b])}
            if with_labels:
                before=float(per_user_ndcg(y[int(rg):int(rg)+1],sc[None,:])[0]); sw=sc.copy(); sw[a],sw[b]=sw[b],sw[a]; after=float(per_user_ndcg(y[int(rg):int(rg)+1],sw[None,:])[0]); rec["delta"]=float(after-before)
            rows.append(rec)
    return pd.DataFrame(rows)

SEMANTIC_FEATURE_NAMES=[
    "relprob_a_mean","relprob_b_mean","relprob_diff_mean","relprob_diff_std",
    "score_a_mean","score_b_mean","score_diff_mean","score_diff_std",
    "pair_pref_mean","pair_pref_std","pair_consistency_mean","pair_vote_frac","rel_vote_frac",
    "hybrid_margin_abs","position_norm","membership","readiness_diff","consensus_weight_diff","module_level_diff","semantic_uncertainty"
]

def semantic_feature_table(tab,rel_members,score_members,pair_forward_members,pair_reverse_members):
    if len(tab)==0:
        out=tab.copy()
        for c in SEMANTIC_FEATURE_NAMES: out[c]=np.nan
        return out
    rl=tab.row_local.to_numpy(int); aa=tab.a.to_numpy(int); bb=tab.b.to_numpy(int)
    ra=np.stack([q[rl,aa] for q in rel_members],axis=1); rb=np.stack([q[rl,bb] for q in rel_members],axis=1); rd=rb-ra
    sa=np.stack([q[rl,aa] for q in score_members],axis=1); sb=np.stack([q[rl,bb] for q in score_members],axis=1); sd=sb-sa
    prefs=[]; cons=[]
    for pf,pr in zip(pair_forward_members,pair_reverse_members):
        d=.5*(np.asarray(pf,float)+(1.0-np.asarray(pr,float))); prefs.append(d); cons.append(1.0-np.abs(np.asarray(pf,float)-(1.0-np.asarray(pr,float))))
    pref=np.stack(prefs,axis=1); con=np.stack(cons,axis=1)
    rg=tab.row_global.to_numpy(int)
    feats=np.column_stack([
        ra.mean(1),rb.mean(1),rd.mean(1),rd.std(1),sa.mean(1),sb.mean(1),sd.mean(1),sd.std(1),
        pref.mean(1),pref.std(1),con.mean(1),(pref>.5).mean(1),(rd>0).mean(1),np.abs(tab.margin.to_numpy(float)),
        tab.position.to_numpy(float)/5.0,tab.membership.to_numpy(float),
        MODULE_READINESS[rg,bb]-MODULE_READINESS[rg,aa],CONS_W[bb]-CONS_W[aa],(MODULE_LEVEL[bb]-MODULE_LEVEL[aa])/4.0,
        pref.std(1)+0.50*sd.std(1)+0.25*rd.std(1)
    ]).astype(np.float32)
    out=tab.copy()
    for j,c in enumerate(SEMANTIC_FEATURE_NAMES): out[c]=feats[:,j]
    return out

def _fit_binary_pipe(X,yy):
    yy=np.asarray(yy,int)
    if len(np.unique(yy))<2: return {"constant":float(yy.mean())}
    m=make_pipeline(StandardScaler(),LogisticRegression(C=.50,class_weight="balanced",max_iter=1500,random_state=SEED)); m.fit(X,yy); return m

def _pred_binary(model,X):
    if isinstance(model,dict): return np.full(len(X),float(model["constant"]),float)
    return model.predict_proba(X)[:,1]

def _fit_residual_models(X,delta):
    delta=np.asarray(delta,float); return {"benefit":_fit_binary_pipe(X,delta>1e-12),"harm":_fit_binary_pipe(X,delta<-1e-12),"utility":make_pipeline(StandardScaler(),Ridge(alpha=3.0)).fit(X,delta)}

def _score_residual_models(bundle,X):
    return _pred_binary(bundle["benefit"],X),_pred_binary(bundle["harm"],X),bundle["utility"].predict(X)

def fit_grouped_residual_calibrator(cal_feat):
    X=cal_feat[SEMANTIC_FEATURE_NAMES].to_numpy(np.float32); d=cal_feat.delta.to_numpy(float); groups=cal_feat.row_local.to_numpy(int); uniq=np.unique(groups); ns=min(5,len(uniq)); assert ns>=2
    o_b=np.zeros(len(cal_feat)); o_h=np.zeros(len(cal_feat)); o_u=np.zeros(len(cal_feat)); gkf=GroupKFold(n_splits=ns)
    for tr,va in gkf.split(X,groups=groups):
        b=_fit_residual_models(X[tr],d[tr]); o_b[va],o_h[va],o_u[va]=_score_residual_models(b,X[va])
    full=_fit_residual_models(X,d); out=cal_feat.copy(); out["p_benefit"]=o_b; out["p_harm"]=o_h; out["utility_pred"]=o_u
    return full,out

def score_residual_calibrator(bundle,feat):
    if len(feat)==0: return feat.copy()
    X=feat[SEMANTIC_FEATURE_NAMES].to_numpy(np.float32); pb,ph,uu=_score_residual_models(bundle,X); out=feat.copy(); out["p_benefit"]=pb; out["p_harm"]=ph; out["utility_pred"]=uu; return out

def family_mask(tab,family):
    if family=="ORDER":
        return ((tab.position>=2)&(tab.position<=4)&(tab.pair_pref_mean>.5)&(tab.pair_vote_frac>=.5)).to_numpy(bool)
    if family=="MEMBERSHIP":
        return ((tab.position==5)&(tab.pair_pref_mean>.5)&(tab.pair_vote_frac>=1.0)&(tab.relprob_diff_mean>0)&(tab.score_diff_mean>0)).to_numpy(bool)
    raise ValueError(family)

def family_economics(tab,family):
    q=tab[family_mask(tab,family)] if len(tab) else tab; d=q.delta.to_numpy(float) if len(q) else np.array([],float)
    return {"gain":float(np.mean(d[d>0])) if np.any(d>0) else (.025 if family=="ORDER" else .10),"harm":float(np.mean(-d[d<0])) if np.any(d<0) else (.025 if family=="ORDER" else .10)}

def add_family_risk(tab,econ):
    out=tab.copy(); exp=out.p_benefit.to_numpy(float)*float(econ["gain"])-out.p_harm.to_numpy(float)*float(econ["harm"]); unc=out.semantic_uncertainty.to_numpy(float)
    out["risk_score"]=0.50*out.utility_pred.to_numpy(float)+0.50*exp-0.25*unc*float(econ["harm"])
    return out

def _dedupe_users(tab):
    if len(tab)==0: return tab.copy()
    return tab.sort_values(["risk_score","p_benefit"],ascending=[False,False],kind="mergesort").drop_duplicates("row_local",keep="first")

def _bootstrap_positive_rate(d,seed):
    d=np.asarray(d,float)
    if len(d)==0: return 0.0
    rng=np.random.default_rng(int(seed)); z=np.empty(POLICY_BOOTSTRAPS,bool)
    for i in range(POLICY_BOOTSTRAPS): z[i]=float(rng.choice(d,size=len(d),replace=True).sum())>0
    return float(z.mean())

def derive_family_policy(cal_oof,n_users,family,seed):
    econ=family_economics(cal_oof,family); z=add_family_risk(cal_oof,econ); z=z[family_mask(z,family)&(z.risk_score>0)].copy(); z=_dedupe_users(z)
    maxcov=ORDER_MAX_COVERAGE if family=="ORDER" else MEMBERSHIP_MAX_COVERAGE; target=ORDER_CAL_PRECISION if family=="ORDER" else MEMBERSHIP_CAL_PRECISION; boot_target=ORDER_BOOT_POS if family=="ORDER" else MEMBERSHIP_BOOT_POS
    maxk=max(1,int(math.ceil(maxcov*n_users))); z=z.head(maxk).reset_index(drop=True); best=None
    for k in range(MIN_POLICY_ACTIONS_CAL,min(maxk,len(z))+1):
        q=z.iloc[:k]; d=q.delta.to_numpy(float); ben=int(np.sum(d>1e-12)); harm=int(np.sum(d<-1e-12)); prec=float(ben/max(1,ben+harm)); inc=float(d.sum()/max(n_users,1)); boot=_bootstrap_positive_rate(d,seed+97*k)
        if prec>=target and inc>0 and boot>=boot_target: best=(k,prec,inc,boot)
    if best is None: return {"family":family,"coverage":0.0,"cal_actions":0,"cal_precision":0.0,"cal_increment":0.0,"boot_positive":0.0,"economics":econ}
    k,prec,inc,boot=best; return {"family":family,"coverage":float(k/max(n_users,1)),"cal_actions":int(k),"cal_precision":float(prec),"cal_increment":float(inc),"boot_positive":float(boot),"economics":econ}

def select_family(scored,n_users,cfg,family):
    if float(cfg.get("coverage",0))<=0 or len(scored)==0: return scored.iloc[:0].copy()
    z=add_family_risk(scored,cfg["economics"]); z=z[family_mask(z,family)&(z.risk_score>0)].copy(); z=_dedupe_users(z); k=max(1,int(math.ceil(float(cfg["coverage"])*n_users))); return z.head(k).reset_index(drop=True)

def combine_actions(order_sel,member_sel):
    if len(order_sel)==0: return member_sel.copy()
    if len(member_sel)==0: return order_sel.copy()
    z=pd.concat([order_sel,member_sel],ignore_index=True); return _dedupe_users(z).reset_index(drop=True)

def apply_actions(reference_scores,selected):
    out=np.asarray(reference_scores,np.float32).copy()
    for r in selected.itertuples(index=False):
        i=int(r.row_local); a=int(r.a); b=int(r.b); out[i,a],out[i,b]=out[i,b],out[i,a]
    return out

def eval_selected(reference,rows_global,selected,seed):
    rows_global=np.asarray(rows_global,int); cand=apply_actions(reference["hybrid"],selected); base=per_user_ndcg(y[rows_global],reference["hybrid"]); cur=per_user_ndcg(y[rows_global],cand); inc=cur-base; touched=np.zeros(len(rows_global),bool)
    if len(selected): touched[selected.row_local.to_numpy(int)]=True
    td=inc[touched]; ben=int(np.sum(td>1e-12)); harm=int(np.sum(td<-1e-12)); tie=int(np.sum(np.abs(td)<=1e-12)); den=max(1,ben+harm); ci=cluster_boot_ci(inc,train.iloc[rows_global].user_id.to_numpy(),seed)
    return {"increment_vs_hybrid34":float(inc.mean()),"ci_low":ci[0],"ci_high":ci[1],"actions":int(touched.sum()),"benefit":ben,"harm":harm,"tie":tie,"precision":float(ben/den),"candidate_scores":cand,"increment_vector":inc}

# Synthetic Stage-C contracts.
_syn=pd.DataFrame({"row_local":[0,0,1],"position":[2,5,5],"pair_pref_mean":[.6,.7,.7],"pair_vote_frac":[.5,1.,1.],"relprob_diff_mean":[.1,.2,-.1],"score_diff_mean":[.1,.2,.2],"risk_score":[.1,.2,.3],"p_benefit":[.7,.8,.8]})
assert family_mask(_syn,"ORDER").tolist()==[True,False,False]
assert family_mask(_syn,"MEMBERSHIP").tolist()==[False,True,False]
print("Stage C frozen residual calibration ready; features",len(SEMANTIC_FEATURE_NAMES))


Stage C frozen residual calibration ready; features 20


## Curriculum world runner and fresh disjoint AA/AB/AC confirmation

Each world uses an 80/20 fit/calibration split inside the outer-train users. Stage A/B gradient updates use **fit labels only**. Calibration labels only train/evaluate the tiny Stage-C calibrator through grouped cross-fitting. AA/AB/AC validation labels are untouched until the precommitted SAFE/ATTACK predictions are scored.


In [19]:
def split_fit_cal(rows,seed):
    rows=np.asarray(rows,int); sss=StratifiedShuffleSplit(n_splits=1,test_size=CAL_FRACTION,random_state=int(seed)); tr,ca=next(sss.split(np.zeros(len(rows)),user_strata_from_y(y[rows]))); return rows[tr],rows[ca]

def make_aaabac_worlds():
    sk=StratifiedKFold(n_splits=10,shuffle=True,random_state=V381_WORLD_SEED); folds=list(sk.split(np.zeros(N_TRAIN),user_strata_from_y(y))); picks={"AA":1,"AB":4,"AC":7}; out={}
    for name,fi in picks.items():
        va=np.asarray(folds[fi][1],int); tr=np.setdiff1d(np.arange(N_TRAIN,dtype=int),va,assume_unique=True); out[name]=(tr,va)
    assert not set(out["AA"][1]).intersection(out["AB"][1]); assert not set(out["AA"][1]).intersection(out["AC"][1]); assert not set(out["AB"][1]).intersection(out["AC"][1])
    return out

def score_pair_datasets(model,tab):
    if len(tab)==0: return np.zeros(0),np.zeros(0)
    f=PairPreferenceDataset(tab,reverse=False,with_labels=False); r=PairPreferenceDataset(tab,reverse=True,with_labels=False); return _stageb_predict(model,f),_stageb_predict(model,r)

def run_semantic_seed(fit_rows,cal_rows,val_rows,hard_train,cal_hard,cal_tab,val_tab,seed,tag,stagea_train_ds,stagea_cal_ds,stagea_val_ds,stageb_train_f,stageb_train_r):
    model_a,hist_a=train_stagea(stagea_train_ds,seed,f"{tag}-A"); cal_rel,cal_score=stagea_matrix_from_dataset(model_a,stagea_cal_ds,len(cal_rows)); val_rel,val_score=stagea_matrix_from_dataset(model_a,stagea_val_ds,len(val_rows))
    cal_truth=y[cal_rows]; try_auc=roc_auc_score((cal_truth>0).reshape(-1),cal_rel.reshape(-1)); stagea_diag={"stage":"A_DIAG","tag":str(tag),"seed":int(seed),"cal_relevance_auc":float(try_auc),"cal_score_mae":float(mean_absolute_error(cal_truth.reshape(-1),cal_score.reshape(-1))),"cal_standalone_ndcg":float(ndcg5(cal_truth,cal_score))}
    core_a=_core_model(model_a); enc_state={k:v.detach().cpu().clone() for k,v in core_a.encoder.state_dict().items()}; del model_a,core_a; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    model_b,hist_b=train_stageb(stageb_train_f,stageb_train_r,enc_state,seed+17,f"{tag}-B"); cal_pf,cal_pr=score_pair_datasets(model_b,cal_tab); val_pf,val_pr=score_pair_datasets(model_b,val_tab)
    if len(cal_hard):
        hf=PairPreferenceDataset(cal_hard,False,False); hr=PairPreferenceDataset(cal_hard,True,False); pf=_stageb_predict(model_b,hf); pr=_stageb_predict(model_b,hr); pp=.5*(pf+(1-pr)); pair_auc=float(roc_auc_score(cal_hard.label.to_numpy(int),pp)) if len(np.unique(cal_hard.label))>1 else float("nan")
    else: pair_auc=float("nan")
    stageb_diag={"stage":"B_DIAG","tag":str(tag),"seed":int(seed),"cal_pair_auc":pair_auc,"hard_train_pairs":int(len(hard_train))}
    del model_b,enc_state; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return {"cal_rel":cal_rel,"cal_score":cal_score,"val_rel":val_rel,"val_score":val_score,"cal_pf":cal_pf,"cal_pr":cal_pr,"val_pf":val_pf,"val_pr":val_pr,"history":hist_a+hist_b+[stagea_diag,stageb_diag]}

def run_v381_world(name,outer_train,val_rows):
    seed=_seed_from_text(f"V38.1-{name}-CURRICULUM"); fit_rows,cal_rows=split_fit_cal(outer_train,seed+11)
    fit_oof=crossfit_hybrid_rows(fit_rows,seed+101,f"V381-{name}-FIT"); cal_ref=fit_reference_hybrid(fit_rows,cal_rows,seed+211); val_ref=fit_reference_hybrid(outer_train,val_rows,seed+307)
    hard_train=build_hard_pair_table(fit_rows,fit_oof["hybrid"]); cal_hard=build_hard_pair_table(cal_rows,cal_ref["hybrid"]); cal_tab=build_residual_table(cal_rows,cal_ref["hybrid"],cal_ref["state"],True); val_tab=build_residual_table(val_rows,val_ref["hybrid"],val_ref["state"],True)
    stagea_train_ds=RelevanceGridDataset(fit_rows,with_labels=True); stagea_cal_ds=RelevanceGridDataset(cal_rows,with_labels=False); stagea_val_ds=RelevanceGridDataset(val_rows,with_labels=False); stageb_train_f=PairPreferenceDataset(hard_train,False,True); stageb_train_r=PairPreferenceDataset(hard_train,True,True)
    members=[]; histories=[]
    for sd in SEMANTIC_SEEDS:
        q=run_semantic_seed(fit_rows,cal_rows,val_rows,hard_train,cal_hard,cal_tab,val_tab,int(sd)+seed,name,stagea_train_ds,stagea_cal_ds,stagea_val_ds,stageb_train_f,stageb_train_r); members.append(q); histories.extend(q["history"])
    cal_feat=semantic_feature_table(cal_tab,[q["cal_rel"] for q in members],[q["cal_score"] for q in members],[q["cal_pf"] for q in members],[q["cal_pr"] for q in members]); val_feat=semantic_feature_table(val_tab,[q["val_rel"] for q in members],[q["val_score"] for q in members],[q["val_pf"] for q in members],[q["val_pr"] for q in members])
    residual_bundle,cal_oof=fit_grouped_residual_calibrator(cal_feat); cal_order=derive_family_policy(cal_oof,len(cal_rows),"ORDER",seed+701); cal_member=derive_family_policy(cal_oof,len(cal_rows),"MEMBERSHIP",seed+809); val_scored=score_residual_calibrator(residual_bundle,val_feat)
    order_sel=select_family(val_scored,len(val_rows),cal_order,"ORDER"); member_sel=select_family(val_scored,len(val_rows),cal_member,"MEMBERSHIP"); attack_sel=combine_actions(order_sel,member_sel)
    safe_eval=eval_selected(val_ref,val_rows,order_sel,seed+901); member_eval=eval_selected(val_ref,val_rows,member_sel,seed+902); attack_eval=eval_selected(val_ref,val_rows,attack_sel,seed+903)
    rows=[{"world":name,"policy":"SAFE",**{k:v for k,v in safe_eval.items() if k not in ("candidate_scores","increment_vector")}},
          {"world":name,"policy":"MEMBERSHIP",**{k:v for k,v in member_eval.items() if k not in ("candidate_scores","increment_vector")}},
          {"world":name,"policy":"ATTACK",**{k:v for k,v in attack_eval.items() if k not in ("candidate_scores","increment_vector")}}]
    calrows=[{"world":name,**{k:v for k,v in cal_order.items() if k!="economics"}},{"world":name,**{k:v for k,v in cal_member.items() if k!="economics"}}]
    led=[]
    for pol,sel in (("SAFE",order_sel),("MEMBERSHIP",member_sel),("ATTACK",attack_sel)):
        if len(sel):
            z=sel.copy(); z.insert(0,"policy",pol); z.insert(0,"world",name); led.append(z)
    ledger=pd.concat(led,ignore_index=True) if led else pd.DataFrame()
    return {"summary":rows,"calibration":calrows,"ledger":ledger,"val_feat":val_feat,"histories":histories,"fit_rows":fit_rows,"cal_rows":cal_rows,"val_rows":val_rows,"cal_cfg":{"ORDER":cal_order,"MEMBERSHIP":cal_member},"val_ref":val_ref}

V381_WORLDS=make_aaabac_worlds(); WORLD_ASSIGNMENTS=pd.DataFrame([{"world":w,"n_train":len(tr),"n_val":len(va),"val_uid_min":train.iloc[va].user_id.min(),"val_uid_max":train.iloc[va].user_id.max()} for w,(tr,va) in V381_WORLDS.items()]); display(WORLD_ASSIGNMENTS)
WORLD_DATA={}; summaries=[]; calrows=[]; ledgers=[]; histories=[]
for name,(tr,va) in V381_WORLDS.items():
    print("\n=== V38.1 fresh world",name,"==="); q=run_v381_world(name,tr,va); WORLD_DATA[name]=q; summaries.extend(q["summary"]); calrows.extend(q["calibration"]); histories.extend(q["histories"]); ledgers.append(q["ledger"])
V381_VALIDATION=pd.DataFrame(summaries); V381_CALIBRATION=pd.DataFrame(calrows); V381_TRAIN_HISTORY=pd.DataFrame(histories); V381_VALIDATION_LEDGER=pd.concat([z for z in ledgers if len(z)],ignore_index=True) if any(len(z) for z in ledgers) else pd.DataFrame()
display(V381_VALIDATION); display(V381_CALIBRATION); display(V381_TRAIN_HISTORY)


,world,n_train,n_val,val_uid_min,val_uid_max
0,AA,3600,400,U_0002,U_3999
1,AB,3600,400,U_0011,U_3984
2,AC,3600,400,U_0001,U_3996



=== V38.1 fresh world AA ===


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,world,policy,increment_vs_hybrid34,ci_low,ci_high,actions,benefit,harm,tie,precision
0,AA,SAFE,0.000000,0.000000,0.000000,0,0,0,0,0.000000
1,AA,MEMBERSHIP,0.000000,0.000000,0.000000,0,0,0,0,0.000000
2,AA,ATTACK,0.000000,0.000000,0.000000,0,0,0,0,0.000000
3,AB,SAFE,0.000248,0.000026,0.000578,9,5,2,2,0.714286
4,AB,MEMBERSHIP,-0.000340,-0.001840,0.000994,7,2,3,2,0.400000
5,AB,ATTACK,-0.000092,-0.001531,0.001332,16,7,5,4,0.583333
6,AC,SAFE,0.000251,-0.000171,0.000726,8,5,2,1,0.714286
7,AC,MEMBERSHIP,0.000000,0.000000,0.000000,0,0,0,0,0.000000
8,AC,ATTACK,0.000251,-0.000175,0.000746,8,5,2,1,0.714286


,world,family,coverage,cal_actions,cal_precision,cal_increment,boot_positive
0,AA,ORDER,0.000000,0,0.0,0.000000,0.000000
1,AA,MEMBERSHIP,0.000000,0,0.0,0.000000,0.000000
2,AB,ORDER,0.020833,15,0.7,0.000145,0.995000
3,AB,MEMBERSHIP,0.015278,11,0.8,0.000306,0.846667
4,AC,ORDER,0.019444,14,0.7,0.000112,0.891667
5,AC,MEMBERSHIP,0.000000,0,0.0,0.000000,0.000000


,stage,tag,seed,epoch,train_loss,trainable_params,cal_relevance_auc,cal_score_mae,cal_standalone_ndcg,cal_pair_auc,hard_train_pairs
0,A,AA-A,665559064,1.0,0.952745,NaN,NaN,NaN,NaN,NaN,NaN
1,B,AA-B,665559081,1.0,1.666322,42528001.0,NaN,NaN,NaN,NaN,NaN
2,A_DIAG,AA,665559064,NaN,NaN,NaN,0.715920,0.214659,0.601287,NaN,NaN
3,B_DIAG,AA,665559064,NaN,NaN,NaN,NaN,NaN,NaN,0.608516,30099.0
4,A,AA-A,665559622,1.0,0.953878,NaN,NaN,NaN,NaN,NaN,NaN
5,B,AA-B,665559639,1.0,1.673149,42528001.0,NaN,NaN,NaN,NaN,NaN
6,A_DIAG,AA,665559622,NaN,NaN,NaN,0.711026,0.226164,0.600538,NaN,NaN
7,B_DIAG,AA,665559622,NaN,NaN,NaN,NaN,NaN,NaN,0.592938,30099.0
8,A,AB-A,10505370,1.0,0.945631,NaN,NaN,NaN,NaN,NaN,NaN
9,B,AB-B,10505387,1.0,1.667244,42528001.0,NaN,NaN,NaN,NaN,NaN


## Fresh promotion gate

`SAFE` is the order-only policy. `ATTACK` is the precommitted one-action-per-user union of order + membership specialists. AA/AB/AC may only accept or reject these frozen policy families; they never tune XLM-R epochs, probabilities, pair construction, coverage rules, or residual feature definitions.


In [20]:
def policy_gate(policy):
    q=V381_VALIDATION[V381_VALIDATION.policy==policy].copy(); vals=q.increment_vs_hybrid34.to_numpy(float); prec=q.precision.to_numpy(float)
    if len(q)!=3: return False,{}
    stats={"mean":float(vals.mean()),"worst":float(vals.min()),"precision_worst":float(prec.min()),"positive_worlds":int(np.sum(vals>0)),"actions_total":int(q.actions.sum())}
    if policy=="SAFE": ok=bool(stats["positive_worlds"]==3 and stats["mean"]>=SAFE_MIN_MEAN and stats["worst"]>=SAFE_MIN_WORST and stats["precision_worst"]>=SAFE_MIN_PRECISION)
    else: ok=bool(stats["positive_worlds"]==3 and stats["mean"]>=ATTACK_MIN_MEAN and stats["worst"]>=ATTACK_MIN_WORST and stats["precision_worst"]>=ATTACK_MIN_PRECISION)
    return ok,stats
SAFE_GATE,SAFE_STATS=policy_gate("SAFE"); ATTACK_GATE,ATTACK_STATS=policy_gate("ATTACK"); DEV_WINNER="ATTACK" if ATTACK_GATE else ("SAFE" if SAFE_GATE else None); DEV_GATE=DEV_WINNER is not None
V381_GATE=pd.DataFrame([{"policy":"SAFE","gate":SAFE_GATE,**SAFE_STATS},{"policy":"ATTACK","gate":ATTACK_GATE,**ATTACK_STATS}]); display(V381_GATE); print("DEV_WINNER",DEV_WINNER,"DEV_GATE",DEV_GATE)


,policy,gate,mean,worst,precision_worst,positive_worlds,actions_total
0,SAFE,False,0.000166,0.000000,0.0,2,17
1,ATTACK,False,0.000053,-0.000092,0.0,1,24


DEV_WINNER None DEV_GATE False


## Sealed template-disjoint D and chronology rejection checks

These expensive checks execute only after AA/AB/AC promote a policy. They rerun the **same** Stage-A → Stage-B → Stage-C curriculum from the pretrained XLM-R checkpoint using only their own training-side labels. They may reject a winner but may not change the recipe.


In [21]:
def sealed_d_bucket(group):
    h=hashlib.sha1(f"{SEALED_D_SALT}::{group}".encode("utf-8")).hexdigest(); return int(h[:12],16)%10
SEALED_D_MASK=np.asarray([sealed_d_bucket(g)==0 for g in TRAIN_GROUP],bool); SEALED_D_TR=np.flatnonzero(~SEALED_D_MASK); SEALED_D_VA=np.flatnonzero(SEALED_D_MASK); assert not set(TRAIN_GROUP[SEALED_D_TR]).intersection(set(TRAIN_GROUP[SEALED_D_VA]))
CHRONO_TR=np.arange(0,3600,dtype=int); CHRONO_VA=np.arange(3600,4000,dtype=int)
SEALED_ROWS=[]; SEALED_LEDGERS=[]
if DEV_GATE:
    for name,tr,va in (("SEALED_D",SEALED_D_TR,SEALED_D_VA),("CHRONOLOGY",CHRONO_TR,CHRONO_VA)):
        print("\n===",name,"rejection ==="); q=run_v381_world(name,tr,va); row=next(r for r in q["summary"] if r["policy"]==DEV_WINNER); SEALED_ROWS.append(row); SEALED_LEDGERS.append(q["ledger"])
V381_SEALED=pd.DataFrame(SEALED_ROWS); V381_SEALED_LEDGER=pd.concat([z for z in SEALED_LEDGERS if len(z)],ignore_index=True) if any(len(z) for z in SEALED_LEDGERS) else pd.DataFrame(); display(V381_SEALED)
if DEV_GATE:
    d=float(V381_SEALED.loc[V381_SEALED.world=="SEALED_D","increment_vs_hybrid34"].iloc[0]); c=float(V381_SEALED.loc[V381_SEALED.world=="CHRONOLOGY","increment_vs_hybrid34"].iloc[0]); SEALED_GATE=bool(d>=SEALED_D_MIN_INCREMENT and c>=CHRONO_MIN_INCREMENT)
else: SEALED_GATE=False
print("SEALED_GATE",SEALED_GATE)


""


SEALED_GATE False


## Exact Hybrid34 test reconstruction and final curriculum deployment

The public anchor is reconstructed exactly from organizer data. Only if development + sealed gates pass do two fresh curriculum seeds train on a shared 80/20 final fit/calibration split. The final residual calibrator is learned only from that calibration split. Test geometry may reject but may never choose/tune a policy.


In [22]:
# Exact full V30/V31/Hybrid34 reconstruction.
full_intent_tr=crossfit_intentstate_features(y,ORDINAL_GRADES,TRAIN_INTENT,MODULE_READY_BAND[:N_TRAIN],TRAIN_GROUP,_seed_from_text("V31-FULL-INTENTSTATE-CROSSFIT"))
FULL_INTENT_BUNDLE=fit_intentstate_bundle(y,ORDINAL_GRADES,TRAIN_INTENT,MODULE_READY_BAND[:N_TRAIN]); full_intent_te=apply_intentstate_bundle(FULL_INTENT_BUNDLE,TEST_INTENT,MODULE_READY_BAND[N_TRAIN:])
V381_SUPPORT_AUDIT=intentstate_support_audit(FULL_INTENT_BUNDLE,TEST_INTENT,MODULE_READY_BAND[N_TRAIN:])
Xtr_full,_,bounds_full=build_long_matrix_v31(np.arange(N_TRAIN),consensus3_oof,base_oof,meta_oof,full_intent_tr); Xte_full,_,bounds_test=build_long_matrix_v31(np.arange(N_TRAIN,N_TRAIN+N_TEST),consensus3_test,base_test,meta_test,full_intent_te); assert bounds_full==bounds_test
full_rank_seed=_seed_from_text("V30-FULL-RANKER"); model_v30,raw_v30=fit_ranker_predict("lambda",Xtr_full[:,:bounds_full["base"]],Xte_full[:,:bounds_full["base"]],ORDINAL_GRADES,N_TRAIN,full_rank_seed); model_v31,raw_v31=fit_ranker_predict("lambda",Xtr_full[:,:bounds_full["intentstate"]],Xte_full[:,:bounds_full["intentstate"]],ORDINAL_GRADES,N_TRAIN,full_rank_seed)
v30_test=blend_residual(consensus3_test,raw_v30,ALPHA); v31_test=blend_residual(consensus3_test,raw_v31,ALPHA); protected_rows=[TEST_UID_TO_ROW[u] for u in sorted(PROTECTED_TEST_USERS)]
v30_test[protected_rows]=v25_anchor_test[protected_rows]; v31_test[protected_rows]=v25_anchor_test[protected_rows]
full_state=state_codes(v25_anchor_test,v30_test,v31_test); hybrid34_test=v30_test.copy(); hybrid34_test[full_state==3]=v31_test[full_state==3]
v25_top=top5_indices(v25_anchor_test); v30_top=top5_indices(v30_test); v31_top=top5_indices(v31_test); v30_changed_rows=np.flatnonzero(np.any(v25_top!=v30_top,axis=1)); v31_changed_rows=np.flatnonzero(np.any(v30_top!=v31_top,axis=1))
v30_changed_users=set(test.iloc[v30_changed_rows]["user_id"].astype(str)); state_count={s:int(np.sum(full_state==i)) for i,s in enumerate(["S0","S1","S2","S3"])}; hybrid_hash=hashlib.sha256(top5_indices(hybrid34_test).astype(np.int16).tobytes()).hexdigest()
V30_REPRO_OK=bool(v30_changed_users==EXPECTED_V30_CHANGED_USERS and int(np.sum(v25_top[:,0]!=v30_top[:,0]))==EXPECTED_V30_TOP1_CHANGES and int(np.sum([set(a)!=set(b) for a,b in zip(v25_top,v30_top)]))==EXPECTED_V30_MEMBERSHIP_CHANGES)
HYBRID_REPRO_OK=bool(len(v31_changed_rows)==EXPECTED_V31_CHANGED_VS_V30 and int(np.sum(full_state==3))==EXPECTED_HYBRID_OVERLAP and state_count==EXPECTED_STATE_COUNTS and hybrid_hash==EXPECTED_HYBRID_TOP5_SHA256)
V381_REPRO_AUDIT={"v30_reproduction_ok":V30_REPRO_OK,"hybrid34_reproduction_ok":HYBRID_REPRO_OK,"v30_changed_users":len(v30_changed_rows),"v31_changed_vs_v30":len(v31_changed_rows),"state_counts":state_count,"hybrid_top5_sha256":hybrid_hash}
print(json.dumps(V381_REPRO_AUDIT,indent=2));
if not V38_SMOKE: assert V30_REPRO_OK and HYBRID_REPRO_OK


def submission_from_scores(scores):
    out=sample.copy(); out.loc[:,MODULES]=np.asarray(scores,np.float32); assert np.isfinite(out[MODULES].to_numpy()).all(); return out

anchor_scores=hybrid34_test.copy(); safe_scores=anchor_scores.copy(); attack_scores=anchor_scores.copy(); membership_scores=anchor_scores.copy(); FINAL_GATE_PREGEOM=bool(DEV_GATE and SEALED_GATE and V30_REPRO_OK and HYBRID_REPRO_OK)
V381_TEST_TAB=pd.DataFrame(); V381_TEST_SCORED=pd.DataFrame(); TEST_LEDGERS={"SAFE":pd.DataFrame(),"MEMBERSHIP":pd.DataFrame(),"ATTACK":pd.DataFrame()}; TEST_GEOMETRY_AUC=float("nan"); GEOMETRY_GATE=False

if FINAL_GATE_PREGEOM:
    all_train=np.arange(N_TRAIN,dtype=int); fit_rows,cal_rows=split_fit_cal(all_train,_seed_from_text("V38.1-FINAL-CAL-SPLIT")); fit_oof=crossfit_hybrid_rows(fit_rows,_seed_from_text("V38.1-FINAL-HYBRID-OOF"),"V381-FINAL-FIT"); cal_ref=fit_reference_hybrid(fit_rows,cal_rows,_seed_from_text("V38.1-FINAL-CAL-REF"))
    hard_train=build_hard_pair_table(fit_rows,fit_oof["hybrid"]); cal_tab=build_residual_table(cal_rows,cal_ref["hybrid"],cal_ref["state"],True); test_rows_global=np.arange(N_TRAIN,N_TRAIN+N_TEST,dtype=int); V381_TEST_TAB=build_residual_table(test_rows_global,hybrid34_test,full_state,False)
    stagea_train_ds=RelevanceGridDataset(fit_rows,True); stagea_cal_ds=RelevanceGridDataset(cal_rows,False); stagea_test_ds=RelevanceGridDataset(test_rows_global,False); stageb_train_f=PairPreferenceDataset(hard_train,False,True); stageb_train_r=PairPreferenceDataset(hard_train,True,True)
    cal_rel=[]; cal_score=[]; test_rel=[]; test_score=[]; cal_pf=[]; cal_pr=[]; test_pf=[]; test_pr=[]
    for sd in SEMANTIC_SEEDS:
        model_a,_=train_stagea(stagea_train_ds,int(sd)+_seed_from_text("V381-FINAL-A"),f"FINAL-{sd}-A"); cr,cs=stagea_matrix_from_dataset(model_a,stagea_cal_ds,len(cal_rows)); trr,tss=stagea_matrix_from_dataset(model_a,stagea_test_ds,N_TEST); core_a=_core_model(model_a); enc_state={k:v.detach().cpu().clone() for k,v in core_a.encoder.state_dict().items()}; del model_a,core_a; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        model_b,_=train_stageb(stageb_train_f,stageb_train_r,enc_state,int(sd)+_seed_from_text("V381-FINAL-B"),f"FINAL-{sd}-B"); cpf,cpr=score_pair_datasets(model_b,cal_tab); tpf,tpr=score_pair_datasets(model_b,V381_TEST_TAB); del model_b,enc_state; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        cal_rel.append(cr); cal_score.append(cs); test_rel.append(trr); test_score.append(tss); cal_pf.append(cpf); cal_pr.append(cpr); test_pf.append(tpf); test_pr.append(tpr)
    cal_feat=semantic_feature_table(cal_tab,cal_rel,cal_score,cal_pf,cal_pr); V381_TEST_FEAT=semantic_feature_table(V381_TEST_TAB,test_rel,test_score,test_pf,test_pr); residual_bundle,cal_oof=fit_grouped_residual_calibrator(cal_feat); cfg_order=derive_family_policy(cal_oof,len(cal_rows),"ORDER",_seed_from_text("V381-FINAL-ORDER")); cfg_member=derive_family_policy(cal_oof,len(cal_rows),"MEMBERSHIP",_seed_from_text("V381-FINAL-MEMBER")); V381_TEST_SCORED=score_residual_calibrator(residual_bundle,V381_TEST_FEAT)
    order_sel=select_family(V381_TEST_SCORED,N_TEST,cfg_order,"ORDER"); member_sel=select_family(V381_TEST_SCORED,N_TEST,cfg_member,"MEMBERSHIP"); attack_sel=combine_actions(order_sel,member_sel); TEST_LEDGERS={"SAFE":order_sel,"MEMBERSHIP":member_sel,"ATTACK":attack_sel}
    safe_scores=apply_actions(hybrid34_test,order_sel); membership_scores=apply_actions(hybrid34_test,member_sel); attack_scores=apply_actions(hybrid34_test,attack_sel)
    for cand in (safe_scores,membership_scores,attack_scores):
        changed=np.any(top5_indices(cand)!=top5_indices(hybrid34_test),axis=1); assert not np.any(changed & (full_state!=0)); assert int(np.sum(top5_indices(cand)[:,0]!=top5_indices(hybrid34_test)[:,0]))==0; assert not np.any(changed[protected_rows])
    # Label-free semantic geometry shift check: fresh AA/AB/AC feature surfaces vs test feature surface.
    dev_geom=pd.concat([WORLD_DATA[w]["val_feat"][SEMANTIC_FEATURE_NAMES] for w in ("AA","AB","AC")],ignore_index=True); test_geom=V381_TEST_FEAT[SEMANTIC_FEATURE_NAMES].reset_index(drop=True); gx=pd.concat([dev_geom,test_geom],ignore_index=True).replace([np.inf,-np.inf],np.nan).fillna(0.0); gy=np.r_[np.zeros(len(dev_geom),int),np.ones(len(test_geom),int)]; cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=_seed_from_text("V381-GEOM")); gp=np.zeros(len(gx),float)
    for tr,va in cv.split(gx,gy):
        clf=make_pipeline(StandardScaler(),LogisticRegression(C=.5,max_iter=1000,random_state=SEED)); clf.fit(gx.iloc[tr],gy[tr]); gp[va]=clf.predict_proba(gx.iloc[va])[:,1]
    TEST_GEOMETRY_AUC=float(roc_auc_score(gy,gp)); GEOMETRY_GATE=bool(TEST_GEOMETRY_AUC<=TEST_GEOMETRY_AUC_MAX)
else:
    V381_TEST_FEAT=pd.DataFrame(); cfg_order={}; cfg_member={}

FINAL_GATE=bool(FINAL_GATE_PREGEOM and GEOMETRY_GATE); selected_scores=attack_scores if DEV_WINNER=="ATTACK" else safe_scores; primary_scores=selected_scores.copy() if FINAL_GATE else hybrid34_test.copy()
SUBMISSIONS={"anchor":submission_from_scores(hybrid34_test),"safe":submission_from_scores(safe_scores),"membership":submission_from_scores(membership_scores),"attack":submission_from_scores(attack_scores),"primary":submission_from_scores(primary_scores)}
SUBMISSIONS["anchor"].to_csv(OUT_DIR/"submission_v38_1_anchor_hybrid34_reconstructed.csv",index=False); SUBMISSIONS["safe"].to_csv(OUT_DIR/"submission_v38_1_DIAGNOSTIC_order_safe.csv",index=False); SUBMISSIONS["membership"].to_csv(OUT_DIR/"submission_v38_1_DIAGNOSTIC_membership_attack.csv",index=False); SUBMISSIONS["attack"].to_csv(OUT_DIR/"submission_v38_1_DIAGNOSTIC_combined_attack.csv",index=False); SUBMISSIONS["primary"].to_csv(OUT_DIR/"submission_v38_1_primary.csv",index=False)
if not FINAL_GATE: assert np.array_equal(SUBMISSIONS["primary"][MODULES].to_numpy(),SUBMISSIONS["anchor"][MODULES].to_numpy())
print("TEST_GEOMETRY_AUC",TEST_GEOMETRY_AUC,"GEOMETRY_GATE",GEOMETRY_GATE,"FINAL_GATE",FINAL_GATE,"DEV_WINNER",DEV_WINNER)


{
  "v30_reproduction_ok": true,
  "hybrid34_reproduction_ok": true,
  "v30_changed_users": 77,
  "v31_changed_vs_v30": 60,
  "state_counts": {
    "S0": 897,
    "S1": 26,
    "S2": 43,
    "S3": 34
  },
  "hybrid_top5_sha256": "47f7db52ce8f782b8b393fd120140f1b6f51aeac452a9fabb5794cc9d8ba5e6e"
}
TEST_GEOMETRY_AUC nan GEOMETRY_GATE False FINAL_GATE False DEV_WINNER None


## Artifacts and submission contract

`submission_v38_1_primary.csv` is submit-eligible **only when `FINAL_GATE=True`**. The membership/order diagnostic files are retained for analysis but are never promoted by test geometry or post-hoc inspection.


In [23]:
WORLD_ASSIGNMENTS.to_csv(OUT_DIR/"v38_1_disjoint_worlds.csv",index=False); TOKENIZER_AUDIT.to_csv(OUT_DIR/"v38_1_tokenizer_audit.csv",index=False); V381_VALIDATION.to_csv(OUT_DIR/"v38_1_validation.csv",index=False); V381_CALIBRATION.to_csv(OUT_DIR/"v38_1_calibration_policy.csv",index=False); V381_TRAIN_HISTORY.to_csv(OUT_DIR/"v38_1_training_history.csv",index=False); V381_GATE.to_csv(OUT_DIR/"v38_1_development_gate.csv",index=False); V381_SEALED.to_csv(OUT_DIR/"v38_1_sealed_validation.csv",index=False); V381_SUPPORT_AUDIT.to_csv(OUT_DIR/"v38_1_intentstate_support_audit.csv",index=False)
if len(V381_VALIDATION_LEDGER): V381_VALIDATION_LEDGER.to_csv(OUT_DIR/"v38_1_validation_interventions.csv",index=False)
if len(V381_SEALED_LEDGER): V381_SEALED_LEDGER.to_csv(OUT_DIR/"v38_1_sealed_interventions.csv",index=False)
if len(V381_TEST_TAB): V381_TEST_TAB.to_csv(OUT_DIR/"v38_1_test_pair_surface.csv",index=False)
if len(V381_TEST_SCORED): V381_TEST_SCORED.to_csv(OUT_DIR/"v38_1_test_scored_pairs.csv",index=False)
for p,z in TEST_LEDGERS.items(): z.to_csv(OUT_DIR/f"v38_1_test_interventions_{p.lower()}.csv",index=False)
with open(OUT_DIR/"v38_1_reproduction_audit.json","w") as f: json.dump(V381_REPRO_AUDIT,f,indent=2)
slot_plan=pd.DataFrame([
    {"file":"submission_v38_1_primary.csv","role":"PRIMARY_ONLY_IF_ALL_GATES_PASS","eligible_to_submit":bool(FINAL_GATE),"note":f"{DEV_WINNER} if AA/AB/AC + sealed + chronology + geometry all pass; otherwise exact Hybrid34 0.66006."},
    {"file":"submission_v38_1_DIAGNOSTIC_combined_attack.csv","role":"DIAGNOSTIC_ONLY","eligible_to_submit":False,"note":"Never bypass preregistered gates."},
    {"file":"submission_v38_1_DIAGNOSTIC_membership_attack.csv","role":"DIAGNOSTIC_ONLY","eligible_to_submit":False,"note":"Membership specialist audit only."},
    {"file":"submission_v38_1_DIAGNOSTIC_order_safe.csv","role":"DIAGNOSTIC_ONLY","eligible_to_submit":False,"note":"Order specialist audit only."},
    {"file":"submission_v38_1_anchor_hybrid34_reconstructed.csv","role":"CURRENT_PUBLIC_ANCHOR_REFERENCE","eligible_to_submit":False,"note":"Exact deterministic public 0.66006 Hybrid34."},
]); slot_plan.to_csv(OUT_DIR/"v38_1_slot_plan.csv",index=False)
summary={"version":"V38.1 Hybrid34-Locked Curriculum XLM-R Semantic Membership & Pairwise Judge","model":str(XLMR_SOURCE),"public_anchor":PUBLIC_HYBRID34_LB,"leader":PUBLIC_LEADER_LB,"leader_gap":PUBLIC_LEADER_GAP,
         "contract":{"stage_a":"17-module relevance BCE + score regression","stage_b":"hard non-tie pair curriculum, upper 6 layers","stage_c":"frozen semantic grouped Logistic/Ridge residual calibration","state":"S0 only","positions":[2,3,4,5],"rank1_locked":True,"semantic_seeds":list(SEMANTIC_SEEDS),"tapt":False,"xe_used":False,"large_model":False},
         "development_winner":DEV_WINNER,"development_gate":bool(DEV_GATE),"sealed_gate":bool(SEALED_GATE),"test_geometry_auc":TEST_GEOMETRY_AUC,"geometry_gate":bool(GEOMETRY_GATE),"final_gate":bool(FINAL_GATE),"development":V381_VALIDATION.to_dict("records"),"sealed":V381_SEALED.to_dict("records"),
         "primary_sha256":hashlib.sha256(SUBMISSIONS["primary"].to_csv(index=False).encode()).hexdigest(),"anchor_sha256":hashlib.sha256(SUBMISSIONS["anchor"].to_csv(index=False).encode()).hexdigest(),
         "submission_instruction":"PRIMARY CLEARED EVERY PRECOMMITTED GATE. Audit intervention ledgers, then primary is the only promoted V38.1 submission." if FINAL_GATE else "NO NEW SUBMISSION. Primary is exact Hybrid34 0.66006; preserve the Kaggle slot."}
with open(OUT_DIR/"v38_1_summary.json","w") as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2)); print(slot_plan.to_string(index=False))


{
  "version": "V38.1 Hybrid34-Locked Curriculum XLM-R Semantic Membership & Pairwise Judge",
  "model": "FacebookAI/xlm-roberta-base",
  "public_anchor": 0.66006,
  "leader": 0.66142,
  "leader_gap": 0.0013600000000000279,
  "contract": {
    "stage_a": "17-module relevance BCE + score regression",
    "stage_b": "hard non-tie pair curriculum, upper 6 layers",
    "stage_c": "frozen semantic grouped Logistic/Ridge residual calibration",
    "state": "S0 only",
    "positions": [
      2,
      3,
      4,
      5
    ],
    "rank1_locked": true,
    "semantic_seeds": [
      271,
      829
    ],
    "tapt": false,
    "xe_used": false,
    "large_model": false
  },
  "development_winner": null,
  "development_gate": false,
  "sealed_gate": false,
  "test_geometry_auc": NaN,
  "geometry_gate": false,
  "final_gate": false,
  "development": [
    {
      "world": "AA",
      "policy": "SAFE",
      "increment_vs_hybrid34": 0.0,
      "ci_low": 0.0,
      "ci_high": 0.0,
      "actions"

### Interpretation contract

- Stage-A standalone NDCG / relevance AUC and Stage-B pair AUC are diagnostics, not promotion metrics.
- AA/AB/AC are the only fresh development cohorts for this curriculum architecture.
- Residual coverage comes only from grouped OOF calibration predictions; there is no post-hoc `P > 0.5/0.6` tuning loop.
- `SAFE` is order-only; `ATTACK` is the fixed order+membership union. Test geometry can only reject.
- If `FINAL_GATE=False`, V38.1 is a scientific negative result and the Kaggle slot must be preserved.
- If `FINAL_GATE=True`, `submission_v38_1_primary.csv` is the only V38.1 file promoted for submission.
